# 서술수준 LoRA — 27B 단독 (코랩 H100 80GB) **v2** (0924 검토 반영)

설계: `claude/설계_서술수준_LoRA_27B단독_왕복게이트_사다리평가_0923.md` · 검토 반영 내역: `claude/검토반영_서술LoRA노트북_v2_0924.md`

| 셀 | STAGE | 역할 | 산출 (`MyDrive/arc_llm/desc_lora/`) |
|---|---|---|---|
| 1 | — | 환경 (STAGE='gen' vLLM / 'train' transformers·peft). 드라이브 미마운트·모델 불완전 다운로드 경고 | — |
| 2 | — | 공용 라이브러리 `desc_lora_lib.py` (프롬프트·facts·기전40·파서·필터·**프로세스 격리 검증**·왕복·엔진) | `/content/desc_lora_lib.py` + 드라이브 사본 |
| 3 | gen | **교사 채굴**: thinking(14K 예산+강제종료) → 필터 → 왕복 게이트 + **무서술 기준선(causal 표기)**. 라운드별 명부 | `desc_teacher_<round>_*.jsonl` |
| 4 | gen | **사후 서술**: 이전 채굴의 정답 코드(재검증) → 서술 → 왕복(코드 비공개) + 무서술 기준선 | `desc_hind_*.jsonl` |
| 5 | gen | **사람 서술(training)** 형식화 → 왕복 + 무서술 기준선 | `desc_human_*.jsonl` |
| 6 | 둘 다 | 조립: training 소속·lib 버전·프롬프트 일치·형식·길이 검증 → 동일규칙 그룹 출처 우선 → 상한3 → 고정 val → 가중(train 만) | `desc_sft_train/val.jsonl`, `val_tids.json` |
| 7 | train | **QLoRA r64** (가중 손실, 성공 microbatch 수로 평균, 옵티마이저·난수·커서 저장) + PEFT 기준 출력 → 키 점검 → `final_vllm` | `lora_desc_q38/final`, `final_vllm` |
| 8 | gen | **사다리 평가**: teacher-forcing NLL 적용증명 → 두 arm 같은 배치 번갈아 → L1~L4, L5 oracle, **L5 2슬롯(정답 비공개 투표)**, L3 recall/precision → **연구 판정 / 채택 판정 분리** | `ladder_*.json` (배치마다 저장) |

**순서**: 셀1(gen)→2→3(SMOKE)→3(본)→4→5→6 → [런타임 재시작] 셀1(train)→2→7 → [런타임 재시작] 셀1(gen)→2→8.

**규율**: evaluation 은 채굴·few-shot 어디에도 안 씀(few-shot = 가상 예 2개 → 검증된 training 서술; 셀6·셀8이 출처 검사) · 학습 입력 = `prompt_desc(t)` · 타겟 = 서술 블록만 · 왕복 통과 + 무서술 실패(causal) 가 정상 샘플, 무서술도 통과한 태스크는 가중 반감 · 사용자 119 서술은 L3 채점에만.

**판정**: 셀8의 연구 판정(ACCEPT)은 "R1 재채굴로 진행할 가치"이지 점수 개선이 아니다. 채택은 2슬롯 정답 태스크의 짝지은 승패(±3 미만 UNDECIDED)와, 그 뒤 동일 시간 예산 리허설의 처리량으로 정한다.

**캐글 이식**: 셀2를 제출 노트북에 그대로. 서술 호출 = `prompt_desc(t)` + `LLM(enable_lora=True, max_lora_rank=64)` + `LoRARequest('desc',1,path)`, 코드·수리 호출은 베이스. `LIB_VERSION` 동일 필수(셀6이 다른 버전 행을 버린다).

**미확인 게이트(G6)**: vLLM 이 `Qwen3_5ForConditionalGeneration` 의 LoRA 를 받는지는 셀8 적재 + teacher-forcing NLL 이 판정. 실패 → `final_vllm` → 병합→FP8 경로. r64 의 메모리는 r16 실측(8K 62GB)의 보장이 아니다 — 셀7 `A.smoke=True` 파일럿으로 피크를 먼저 본다.

In [1]:
# [셀1] 환경 — STAGE 에 따라 설치가 다르다. ★셀2(라이브러리)는 항상 이어서 실행.
#   STAGE='gen'   : 셀3(교사 채굴) · 셀4(사후 서술) · 셀5(사람 서술) · 셀6(조립) · 셀8(사다리 평가)  — vLLM 런타임
#   STAGE='train' : 셀6(조립) · 셀7(QLoRA)                                                        — transformers/peft 런타임
#   두 스택을 한 런타임에 섞지 않는다 (torch 버전 충돌). 단계가 바뀌면 [런타임 재시작] → 셀1 → 셀2.
STAGE = 'gen'            # ★ 'gen' | 'train'

import os, sys, subprocess, shutil
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['ARC_ROOT'] = '/content/ARC-AGI-2'
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print('드라이브 마운트 생략:', e)
DRIVE_ROOT = '/content/drive/MyDrive/arc_llm'
DESC_DIR   = f'{DRIVE_ROOT}/desc_lora'            # ★이 노트북의 모든 산출물
if not os.path.exists('/content/drive/MyDrive'):
    DRIVE_ROOT = '/content/arc_llm_LOCAL'; DESC_DIR = f'{DRIVE_ROOT}/desc_lora'
    print('★★ 드라이브 미마운트 — 산출물이 런타임과 함께 사라진다. 로컬 경로로 진행:', DESC_DIR)
os.makedirs(DESC_DIR, exist_ok=True)
if not os.path.exists('/content/ARC-AGI-2/data/training'):
    subprocess.run('git clone -q --depth 1 https://github.com/arcprize/ARC-AGI-2 /content/ARC-AGI-2', shell=True, check=True)

def _pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *a], capture_output=True, text=True)
    print(('OK ' if r.returncode == 0 else '★FAIL ') + ' '.join(a))
    if r.returncode:
        print((r.stdout + r.stderr)[-1200:])
    return r.returncode == 0

if STAGE == 'gen':
    _pip('vllm>=0.19', 'huggingface_hub', 'hf_transfer', 'safetensors')     # vllm 이 자기 torch/transformers 를 끌고 온다
    _pip('flash-linear-attention') or print('   (fla 없음 — GDN 커널은 vLLM 내장 Triton 사용)')
    import vllm, torch
    print(f'vllm {vllm.__version__} / torch {torch.__version__}')
else:
    _pip('transformers>=5.15', 'accelerate', 'peft', 'bitsandbytes', 'safetensors', 'huggingface_hub', 'hf_transfer')
    _pip('flash-linear-attention') or print('   ★fla 없음 — linear_attention 48층 torch 폴백(느림)')
    _pip('causal-conv1d') or print('   (causal-conv1d 빌드 실패 — 무관)')
    import transformers, peft, bitsandbytes, torch
    print(f'transformers {transformers.__version__} / peft {peft.__version__} / bnb {bitsandbytes.__version__} / torch {torch.__version__}')

import torch
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {torch.cuda.get_device_name(0) if _vram else "없음"} {_vram:.0f}GB')
assert _vram >= 70, '★H100 80GB 런타임이 아니다 — 런타임 유형 확인'

# 모델 (교사=학생=서빙 동일: Qwen/Qwen3.8-27B bf16 표준판, 56GB)
MODEL_ID  = 'Qwen/Qwen3.8-27B'
LOCAL_DIR = '/content/q38_bf16'
def _model_complete():
    idx = f'{LOCAL_DIR}/model.safetensors.index.json'
    if not os.path.exists(f'{LOCAL_DIR}/config.json'):
        return False
    if os.path.exists(idx):
        import json as _j
        need = set(_j.load(open(idx))['weight_map'].values())
        return all(os.path.exists(f'{LOCAL_DIR}/{f}') and os.path.getsize(f'{LOCAL_DIR}/{f}') > 0 for f in need)
    return bool(glob.glob(f'{LOCAL_DIR}/*.safetensors'))
import glob
def ensure_model():
    if _model_complete():
        return
    free = shutil.disk_usage('/content').free / 1e9
    assert free > 65, f'★디스크 부족 {free:.0f}GB < 65GB'
    from huggingface_hub import snapshot_download
    print(f'HF 다운로드 {MODEL_ID} (~56GB)...')
    snapshot_download(MODEL_ID, local_dir=LOCAL_DIR, max_workers=8,
                      allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja', '*.py', '*.model', '*.tiktoken'],
                      ignore_patterns=['*consolidated*', 'original/*'])
print('★셀1 끝 → 셀2(라이브러리). transformers/vllm 이 새로 깔렸으면 [런타임 재시작] 후 셀1부터.')


OK vllm>=0.19 huggingface_hub hf_transfer safetensors
OK flash-linear-attention
vllm 0.30.0 / torch 2.13.0+cu130
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition 102GB
★셀1 끝 → 셀2(라이브러리). transformers/vllm 이 새로 깔렸으면 [런타임 재시작] 후 셀1부터.


In [2]:
# [셀2] 공용 라이브러리 — /content/desc_lora_lib.py 로 쓰고 드라이브에 사본. ★캐글 제출 노트북에도 이 셀을 그대로 넣는다.
import os, shutil
LIB_SRC = r'''# -*- coding: utf-8 -*-
"""
desc_lora_lib.py — 27B 서술수준 LoRA 공용 라이브러리 (코랩 H100 / 캐글 공용)
================================================================================
설계: claude/설계_서술수준_LoRA_27B단독_왕복게이트_사다리평가_0923.md

한 파일이 다음 전부의 단일 원천이다 — 채굴·학습·평가·캐글 서빙이 전부 여기서 import 한다.
  * 데이터 적재 / 격자 문자열
  * facts 2.0 (MEASURED FACTS — 기계 계산 사실, 규칙 힌트 아님)
  * 기전 어휘 40 (KO 사용자 서술 / EN 모델 서술 양쪽 검출)
  * 서술 프롬프트 (교사·학생·서빙 동일 텍스트)  ★한 글자라도 바뀌면 전이 안 됨
  * 서술 블록 파서 (HEAD/MECH/STEPS/PARAM/SIZES)
  * 필터: HEAD 사실 일치 · 기전 ≥2 · 리터럴 파라미터 · 중복
  * 샌드박스 + 왕복 게이트 (서술 → 코드 → train 전쌍 + test)
  * 사람 서술(eval119 / training memo) 파서
  * vLLM 엔진 래퍼 (thinking 2패스, LoRARequest)
  * 어댑터 키 점검 / 리네임
"""
import os, re, json, glob, time, random, signal, hashlib, math
from collections import Counter, deque
import numpy as np

# ------------------------------------------------------------------------------
# 0. 상수
# ------------------------------------------------------------------------------
LIB_VERSION = '0924b'
ARC_ROOT = os.environ.get('ARC_ROOT', '/content/ARC-AGI-2')
EXEC_TIMEOUT = 4

def LOG(*a):
    print(f'[{time.strftime("%H:%M:%S")}]', *a, flush=True)

# ------------------------------------------------------------------------------
# 1. 데이터
# ------------------------------------------------------------------------------
def load_split(split, root=None):
    root = root or ARC_ROOT
    fs = sorted(glob.glob(f'{root}/data/{split}/*.json'))
    out = []
    for f in fs:
        t = json.load(open(f))
        sol = [te.get('output') for te in t.get('test', [])]
        t['_sol'] = sol if (sol and all(s is not None for s in sol)) else None
        out.append((os.path.basename(f)[:-5], t))
    return out

def _g(x):
    return np.asarray(x, dtype=int)

def gs(g):
    return '\n'.join(' '.join(str(int(v)) for v in row) for row in g)

def task_cells(t):
    n = sum(_g(p['input']).size + _g(p['output']).size for p in t['train'])
    return n + sum(_g(te['input']).size for te in t['test'])

def usable(t, max_side=30):
    tr, te = t.get('train', []), t.get('test', [])
    if len(tr) < 2 or not te or t.get('_sol') is None:
        return False
    for p in tr:
        for g in (p['input'], p['output']):
            if len(g) > max_side or len(g[0]) > max_side:
                return False
    return all(len(x['input']) <= max_side and len(x['input'][0]) <= max_side for x in te)

# ------------------------------------------------------------------------------
# 2. facts 2.0 — 기계 계산 사실 (서술개혁 파일럿 ① 그대로: 크기·배경·overwrite 생존·
#    출력⊂입력·색깔별 객체 수·분단선·홀).  규칙 힌트 금지, 항상 같은 구조.
# ------------------------------------------------------------------------------
def _bg(grids):
    c = Counter()
    for g in grids:
        c.update(_g(g).ravel().tolist())
    return c.most_common(1)[0][0]

def _components(a, color):
    """4-연결 단색 성분 개수."""
    m = (a == color)
    seen = np.zeros_like(m, dtype=bool); n = 0
    H, W = a.shape
    for y in range(H):
        for x in range(W):
            if m[y, x] and not seen[y, x]:
                n += 1; q = deque([(y, x)]); seen[y, x] = True
                while q:
                    cy, cx = q.popleft()
                    for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                        ny, nx = cy+dy, cx+dx
                        if 0 <= ny < H and 0 <= nx < W and m[ny, nx] and not seen[ny, nx]:
                            seen[ny, nx] = True; q.append((ny, nx))
    return n

def _holes(a, bg):
    """배경색 영역 중 격자 가장자리에 닿지 않는 것(=홀) 개수."""
    H, W = a.shape
    m = (a == bg); seen = np.zeros_like(m, dtype=bool); holes = 0
    for y in range(H):
        for x in range(W):
            if m[y, x] and not seen[y, x]:
                q = deque([(y, x)]); seen[y, x] = True; touch = False
                while q:
                    cy, cx = q.popleft()
                    if cy in (0, H-1) or cx in (0, W-1):
                        touch = True
                    for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                        ny, nx = cy+dy, cx+dx
                        if 0 <= ny < H and 0 <= nx < W and m[ny, nx] and not seen[ny, nx]:
                            seen[ny, nx] = True; q.append((ny, nx))
                if not touch:
                    holes += 1
    return holes

def _separators(a, bg):
    """한 가지 비배경색으로 꽉 찬 행/열 수."""
    n = 0
    for row in a:
        if len(set(row.tolist())) == 1 and row[0] != bg:
            n += 1
    for col in a.T:
        if len(set(col.tolist())) == 1 and col[0] != bg:
            n += 1
    return n

def _is_subgrid(o, i):
    H, W = o.shape; h, w = i.shape
    if H > h or W > w:
        return False
    for y in range(h - H + 1):
        for x in range(w - W + 1):
            if np.array_equal(i[y:y+H, x:x+W], o):
                return True
    return False

def task_facts(t):
    """구조화 사실 dict — HEAD 검증(L2)과 facts 블록이 같은 값을 쓴다."""
    ins = [p['input'] for p in t['train']]
    bg = _bg(ins)
    same = all(_g(p['input']).shape == _g(p['output']).shape for p in t['train'])
    diff = all(_g(p['input']).shape != _g(p['output']).shape for p in t['train'])
    sub = all(_is_subgrid(_g(p['output']), _g(p['input'])) for p in t['train']) and not same
    per = []
    for p in t['train']:
        i, o = _g(p['input']), _g(p['output'])
        d = {'in': i.shape, 'out': o.shape}
        if i.shape == o.shape:
            d['changed'] = int((i != o).sum())
            nb = (i != bg)
            d['nonbg_kept'] = int(((i == o) & nb).sum()); d['nonbg'] = int(nb.sum())
        cin = {c: _components(i, c) for c in np.unique(i).tolist() if c != bg}
        cout = {c: _components(o, c) for c in np.unique(o).tolist() if c != bg}
        d['obj_in'] = cin; d['obj_out'] = cout
        d['sep_in'] = _separators(i, bg)
        d['holes_in'] = _holes(i, bg)
        per.append(d)
    return {'bg': int(bg), 'size': 'SAME' if same else ('DIFFERENT' if diff else 'MIXED'),
            'subgrid': bool(sub), 'per': per}

def facts_block(t, F=None):
    F = F or task_facts(t)
    L = [f"Output size vs input: {F['size']}. Most common input color (assumed background, may be wrong): {F['bg']}.",
         f"Output is a sub-grid cut from the input: {'yes' if F['subgrid'] else 'no'}."]
    for j, d in enumerate(F['per']):
        s = f"Example {j+1}: input {d['in'][0]}x{d['in'][1]} -> output {d['out'][0]}x{d['out'][1]}"
        if 'changed' in d:
            s += f"; cells changed {d['changed']}/{d['in'][0]*d['in'][1]}; non-background input cells kept {d['nonbg_kept']}/{d['nonbg']}"
        oi = ' '.join(f"c{c}:{n}" for c, n in sorted(d['obj_in'].items()))
        oo = ' '.join(f"c{c}:{n}" for c, n in sorted(d['obj_out'].items()))
        s += f"; objects per color in->out [{oi}] -> [{oo}]"
        s += f"; separator lines {d['sep_in']}; holes {d['holes_in']}"
        L.append(s)
    return "MEASURED FACTS (computed by code, trust them):\n" + '\n'.join(L)

# ------------------------------------------------------------------------------
# 3. 기전 어휘 40 — (KO 키워드: 사용자 서술) / (EN 키워드: 모델 서술)
#    ★vocab.py 원본이 있으면 이 표를 그 표로 교체한다. LOW_INFO 는 L3 채점에서 제외.
# ------------------------------------------------------------------------------
MECHS = {
 'extract':     (['추출', '색출', '분리', '선택'],           ['extract', 'isolate', 'select', 'pick out', 'separate']),
 'outline':     (['외곽선', '외각선', '외곽', '외각', '테두리'], ['outline', 'border', 'contour', 'boundary', 'perimeter']),
 'background':  (['배경'],                                   ['background']),
 'dot':         (['점'],                                     ['dot', 'single cell', 'single-cell', 'isolated cell', 'marker']),
 'line':        (['직선', r'(?<![곽각전])선(?![택정])'],        ['line', 'straight', 'segment']),
 'move':        (['무빙', '이동', '움직', '옮'],               ['move', 'shift', 'translate', 'slide', 'relocate']),
 'rectangle':   (['사각형', '상자', '박스', '정사각'],         ['rectangle', 'rectangular', 'box', 'square']),
 'per_color':   (['색깔별', '색깔 별', '색별', '색상별', '색상 별', '쌕깔별'], ['per color', 'each color', 'by color', 'per-color', 'for every color', 'colour']),
 'rotate':      (['회전', 'rotate', '90', '180', '270'],      ['rotate', 'rotation', 'rotated']),
 'panel':       (['패널', '분할', '분단', '휴전선', '나누기 선', '방을'], ['panel', 'partition', 'divider', 'separator', 'section', 'region divided']),
 'erase':       (['지우', '없앰', '없애', '제거', '삭제', '사라'], ['erase', 'remove', 'delete', 'clear', 'disappear', 'drop']),
 'inside':      (['내부', '안에', '안의', '안을', '안 '],      ['inside', 'interior', 'within', 'enclosed', 'inner']),
 'max_min':     (['가장 큰', '가장 작', '최대', '최소', '큰 것', '작은 것', '큰 놈', '작은 놈'], ['largest', 'smallest', 'biggest', 'maximum', 'minimum', 'longest', 'shortest']),
 'recolor':     (['색변환', '색 변환', '색상 변화', '색깔 변', '변신', '색 바꾸', '반전', '치환', '색깔 넣', '색 추가', '칠하'], ['recolor', 'recolour', 'change color', 'change the color', 'replace color', 'color mapping', 'swap color', 'invert', 'repaint', 'becomes color']),
 'cross':       (['십자'],                                   ['cross', 'plus shape', 'plus-shaped']),
 'count':       (['개수', '갯수', '수 만큼', '세기', '카운트', '카우트', '수에 따라'], ['count', 'number of', 'how many', 'as many as']),
 'grid':        (['그리드', '격자'],                          ['grid cells', 'sub-grid', 'subgrid', 'cell of the grid', 'grid lines', 'lattice', 'blocks of']),
 'sort':        (['정렬', '순서대로', '서열', '순위', '소트', '순서'], ['sort', 'order by', 'rank', 'in order of', 'sequence', 'ordered']),
 'assemble':    (['조립', '합성', '합체', '붙여넣기', '붙이기', '붙임', '재조립', '끼우', '재탄생', '합침', '합병'], ['assemble', 'reassemble', 'combine', 'compose', 'place into', 'fit into', 'paste', 'merge', 'stack']),
 'hole':        (['홀', '구멍', '빈공간', '빈 공간', '빈자리'],  ['hole', 'cavity', 'enclosed empty', 'empty region', 'gap inside']),
 'distance':    (['거리', '간격', '칸 떨어'],                  ['distance', 'apart', 'spacing', 'gap of', 'cells away']),
 'ray':         (['광선', '부딪', '부딛', '꺽', '굴절', '관통', '발사', '직진'], ['ray', 'shoot', 'until it hits', 'until it reaches', 'bounce', 'bend', 'deflect', 'beam', 'stops when']),
 'legend':      (['제어표', '범례', '색상표', '표시', '지정'],   ['legend', 'key', 'control table', 'lookup', 'palette table', 'reference', 'template']),
 'copy':        (['복제', '복사'],                            ['copy', 'copies', 'duplicate', 'replicate', 'clone', 'stamp']),
 'path':        (['경로', '골목', '흐름', '길'],                ['path', 'route', 'corridor', 'trace', 'walk', 'trail']),
 'connect':     (['연결', '이어', '잇', '붙어있', '연달아'],     ['connect', 'link', 'join', 'adjacent', 'touching']),
 'draw_line':   (['선 긋', '선긋', '직선 추가', '그리기', '긋기', '긋고', '그음', '그림'], ['draw a line', 'draw lines', 'draw', 'extend a line', 'extend', 'draw the']),
 'scale':       (['확대', '확장', '키우', '배로'],             ['scale up', 'enlarge', 'upscale', 'expand', 'zoom', 'magnif', 'times larger', 'scaled']),
 'shape_class': (['모양분류', '모양 분류', '모양 일치', '모양별', '같은 모양', '같은모양', '모양 종류', '공통 모양'], ['same shape', 'shape class', 'matching shape', 'classify shapes', 'identical shape', 'shape type', 'by shape']),
 'mirror':      (['미러', '대칭', '반사', '좌우 ', '상하 ', '뒤집'], ['mirror', 'reflect', 'symmetric', 'symmetry', 'flip']),
 'gravity':     (['중력', '떨어', '내려감', '올라감', '흘러내림', '상승', '쌓'], ['gravity', 'fall', 'drop down', 'rise up', 'sink', 'float up', 'pile']),
 'tile':        (['타일', '반복', '패턴', '등간격'],           ['tile', 'tiled', 'repeat', 'tiling', 'periodic', 'pattern']),
 'fill':        (['채움', '채우', '채워', '도포', '메우'],       ['fill', 'flood', 'paint', 'filled']),
 'restore':     (['복원', '복구', '원상'],                     ['restore', 'repair', 'complete the', 'reconstruct', 'missing part']),
 'shrink':      (['축소', '줄이', '작게 만'],                  ['shrink', 'downscale', 'reduce', 'compress', 'condense']),
 'cluster':     (['클러스터', 'kmeans', '군집', '덩어리'],     ['cluster', 'group of', 'grouped']),
 'unique':      (['유일', '하나만', '다른 하나', '비일치'],     ['unique', 'odd one out', 'the only', 'differs from the others', 'different from all']),
 'overwrite':   (['overwrite', 'overwirte', '오버라이트', '오보라이트', '겹쳐쓰기', '덮', '씌우'], ['overwrite', 'overlay', 'on top of', 'overwrites']),
 'fixed':       (['고정', '일치', '그대로', '픽스'],            ['unchanged', 'keep', 'kept', 'fixed', 'stays', 'remain']),
 'surface':     (['표면', '모양', '도형'],                     ['shape', 'object']),
}
LOW_INFO = {'surface', 'fixed', 'overwrite', 'background'}   # 실측 84~99% 등장 → 변별력 없음
MECH_KEYS = list(MECHS.keys())

def detect_mechs_ko(text):
    t = text or ''
    return {k for k, (ko, _) in MECHS.items() if any(re.search(w, t) for w in ko)}

def detect_mechs_en(text):
    t = (text or '').lower()
    return {k for k, (_, en) in MECHS.items() if any(w in t for w in en)}

# ------------------------------------------------------------------------------
# 4. 사람 서술 파서 (eval119 메모 / training 메모)
# ------------------------------------------------------------------------------
def parse_human_memo(text, key_re=r'^(\d+)\.?$'):
    """번호줄(또는 tid 줄)이 새 서술 시작. ##### / //// 줄은 구분선. 반환 {키: {'head':[], 'rule':[]}}."""
    D, cur = {}, None
    for raw in text.replace('\r', '').split('\n'):
        l = raw.strip()
        if not l or re.match(r'^#{3,}', l) or re.match(r'^/{3,}\s*$', l):
            continue
        m = re.match(key_re, l)
        if m:
            cur = m.group(1); D[cur] = {'head': [], 'rule': []}; continue
        if cur is None:
            continue
        d = D[cur]
        if re.search(r'크기|일치|배경|overwrite|overwirte|모양 (다 )?같음|rotate', l) and not d['rule'] and not l.startswith('/'):
            d['head'].append(l)
        else:
            d['rule'].append(l.lstrip('/').strip())
    return D

def human_memo_text(d):
    return ' / '.join(d['head']) + ' :: ' + ' / '.join(x for x in d['rule'] if x)

def eval119_by_tid(memo_path, eval_ids):
    """서술 #n = sorted(evaluation)[n]  (실측 확정: 오프셋 +1, 114/117 일치). 반환 {tid: memo dict}"""
    D = parse_human_memo(open(memo_path, encoding='utf-8', errors='ignore').read())
    out = {}
    for k, d in D.items():
        n = int(k)
        if 0 <= n < len(eval_ids) and (d['head'] or d['rule']):
            out[eval_ids[n]] = d
    return out

def human_mech_set(d):
    return detect_mechs_ko(' '.join(d['head'] + d['rule'])) - LOW_INFO

# ------------------------------------------------------------------------------
# 5. 프롬프트 — ★교사·학생·평가·캐글 서빙 전부 이 함수. 텍스트를 바꾸면 LIB_VERSION 을 올린다.
# ------------------------------------------------------------------------------
MECH_MENU = ', '.join(k for k in MECH_KEYS if k not in LOW_INFO)

DESC_FMT = f"""Describe the rule in EXACTLY this format and nothing else:

HEAD: size=<SAME|DIFFERENT|MIXED>; bg=<the most common input color, as given in MEASURED FACTS>; subgrid=<yes|no>
MECH: <2 to 6 mechanism tags from this closed list: {MECH_MENU}>
STEPS:
1. <one action per step, 2 to 6 steps; every step names WHICH cells/objects it acts on by a selector (largest, the color that appears once, the object touching the border, ...), never by absolute coordinates>
2. ...
PARAM: <the parameters your steps read from the examples (colors, counts, directions) and HOW they are read, e.g. "target color = the color of the single isolated cell">
SIZES: <HxW for Example 1>, <HxW for Example 2>, ...

Rules for a good description: the HEAD must agree with the MEASURED FACTS; MECH tags must come from the list; each STEP is one mechanism applied to a selected set of objects; a programmer must be able to implement it without seeing the examples; SIZES is the output size YOUR rule predicts for each Example, in order. Do not write code."""

def task_block(t):
    parts = []
    for j, p in enumerate(t['train']):
        parts.append(f"Example {j+1} input:\n{gs(p['input'])}\nExample {j+1} output:\n{gs(p['output'])}")
    for j, te in enumerate(t['test']):
        parts.append(f"Test input {j+1}:\n{gs(te['input'])}")
    return '\n\n'.join(parts)

def prompt_desc(t, F=None, fewshot=''):
    """★학생 학습 입력 = 이 함수(fewshot='') 그대로. 교사에게만 fewshot 을 붙인다."""
    return (f"This is an ARC puzzle. Every Example transforms an input grid of digits 0-9 into an output grid by ONE consistent rule.\n\n"
            f"{task_block(t)}\n\n{facts_block(t, F)}\n\n{fewshot}{DESC_FMT}")

def prompt_desc_from_code(t, code, F=None, fewshot=''):
    """사후 서술(hindsight): 정답 코드를 보고 서술. 학습 타겟은 서술만, 프롬프트는 prompt_desc(깨끗한 것)."""
    return (f"This is an ARC puzzle. Every Example transforms an input grid of digits 0-9 into an output grid by ONE consistent rule.\n\n"
            f"{task_block(t)}\n\n{facts_block(t, F)}\n\n"
            f"A verified program that solves this puzzle is:\n```python\n{code.strip()}\n```\n\n"
            f"Explain the rule this program implements, as a human would describe it — mechanisms and selectors, not code. "
            f"Do not mention variables, loops or Python.\n\n{fewshot}{DESC_FMT}")

def prompt_desc_from_memo(t, memo_text, F=None):
    """사람(한국어) 메모 → 형식 서술로 정규화."""
    return (f"This is an ARC puzzle. Every Example transforms an input grid of digits 0-9 into an output grid by ONE consistent rule.\n\n"
            f"{task_block(t)}\n\n{facts_block(t, F)}\n\n"
            f"A human expert wrote this terse Korean note about the rule (their vocabulary: 추출=extract, 외곽선=outline, 무빙=move, "
            f"색깔별=per color, 점=dot, 직선=line, 홀=hole, 제어표=legend, 붙여넣기=assemble, overwrite 일치=cells kept on overlay):\n"
            f"{memo_text}\n\nRewrite it faithfully — same mechanisms, same order — in the format below. Fill PARAM from the examples.\n\n{DESC_FMT}")

def prompt_code(t, desc):
    return (f"This is an ARC puzzle.\n\n{task_block(t)}\n\nThe rule is:\n{desc}\n\n"
            f"Implement it:\n```python\ndef solve(grid):\n    # grid: [[int]] -> return [[int]]\n```\n"
            f"Use ONLY plain Python and numpy (np). Do NOT import anything.\n"
            f"IMPORTANT: grid is the raw 2D list exactly as shown — no header row, no size row; grid[0] is the first picture row.\n"
            f"It must reproduce EVERY Example output exactly. No comments.\nOutput ONLY the ```python code block.")

# 형식 few-shot (교사 전용) — ★어느 ARC 태스크에도 대응하지 않는 가상의 예 2개 (eval 출처 금지: 평가 오염).
#   검증된 training 서술이 쌓이면 dynamic_fewshot 이 그것으로 교체한다 (출처 tid 기록).
FEWSHOT_BANK = [
 {'tid': None, 'mechs': {'panel', 'count', 'recolor', 'max_min'},
  'text': "HEAD: size=SAME; bg=0; subgrid=no\nMECH: panel, count, max_min, recolor\nSTEPS:\n1. Split the grid into panels at the full-length separator lines.\n2. Count the non-background cells in each panel.\n3. Recolor every cell of the panel with the largest count to the separator color; leave the other panels unchanged.\nPARAM: separator color = the color of the full-length lines; target color = separator color\nSIZES: 9x9, 9x13, 13x9"},
 {'tid': None, 'mechs': {'extract', 'shape_class', 'copy', 'gravity'},
  'text': "HEAD: size=DIFFERENT; bg=0; subgrid=no\nMECH: extract, shape_class, copy, gravity\nSTEPS:\n1. Extract every object as a connected group of same-colored cells.\n2. Keep only the objects whose shape matches the single object that touches the top border.\n3. Drop each kept object straight down until it rests on the bottom edge or on another object.\n4. The output is the bounding box of all kept objects after they settle.\nPARAM: reference shape = the object touching the top border; matching = identical shape up to translation\nSIZES: 4x7, 5x5, 3x9"},
]

def pick_fewshot(target_mechs=None, k=2, rng=None, bank=None):
    """대상과 기전이 겹치지 않는 few-shot 을 고른다 (v8 앵커 교훈). 반환 (텍스트, 출처 tid 목록)."""
    rng = rng or random
    bank = list(bank or FEWSHOT_BANK); rng.shuffle(bank)
    cand = sorted(bank, key=lambda e: len(e['mechs'] & (target_mechs or set())))
    chosen = cand[:k]
    txt = 'Two examples of the FORMAT on other puzzles (different mechanisms — do not copy their content):\n\n' + \
          '\n\n'.join(e['text'] for e in chosen) + '\n\n'
    return txt, [e.get('tid') for e in chosen]

def dynamic_fewshot_bank(rows, max_n=40):
    """검증된 training 서술(desc_*.jsonl 행)로 few-shot 은행. 태스크당 1개, 기전집합이 다양하도록."""
    bank, seen = [], set()
    for r in rows:
        ms = frozenset(r.get('mechs', []))
        if r['tid'] in seen or ms in {b['mechs'] for b in bank}:
            continue
        seen.add(r['tid']); bank.append({'tid': r['tid'], 'mechs': set(ms), 'text': r['completion']})
        if len(bank) >= max_n:
            break
    return bank or None

# ------------------------------------------------------------------------------
# 6. 서술 블록 파서 + 필터
# ------------------------------------------------------------------------------
_SEC = ['HEAD', 'MECH', 'STEPS', 'PARAM', 'SIZES']

def parse_desc(txt):
    """반환 dict: head, mech(set), steps(list), param, sizes(list|None), block(정규화 텍스트) / None 이면 형식 실패."""
    if not txt:
        return None
    t = re.sub(r'<think>.*?</think>', '', txt, flags=re.S).split('</think>')[-1]
    t = re.sub(r'```.*?```', '', t, flags=re.S)
    pos = {}
    for s in _SEC:
        m = re.search(rf'(?m)^\s*{s}\s*:', t)
        if m:
            pos[s] = m.start()
    if not all(s in pos for s in ('HEAD', 'MECH', 'STEPS', 'SIZES')):
        return None
    order = sorted(pos.items(), key=lambda x: x[1])
    sec = {}
    for i, (s, p) in enumerate(order):
        end = order[i+1][1] if i+1 < len(order) else len(t)
        body = t[p:end]; body = body.split(':', 1)[1].strip()
        sec[s] = body
    head = ' '.join(sec['HEAD'].split())
    hm = re.search(r'size\s*=\s*(SAME|DIFFERENT|MIXED)', head, re.I)
    bm = re.search(r'bg\s*=\s*(\d)', head, re.I)
    sm = re.search(r'subgrid\s*=\s*(yes|no)', head, re.I)
    mech = {w.strip().lower().replace('-', '_') for w in re.split(r'[,\s]+', sec['MECH']) if w.strip()}
    mech = {m for m in mech if m in MECHS}
    steps = [' '.join(l.split()) for l in sec['STEPS'].split('\n') if re.match(r'^\s*\d+[.)]', l)]
    steps = [re.sub(r'^\d+[.)]\s*', '', s) for s in steps]
    sizes = None
    pr = re.findall(r'(\d{1,2})\s*[xX×*]\s*(\d{1,2})', sec['SIZES'].split('\n')[0])
    if pr:
        sizes = [(int(a), int(b)) for a, b in pr]
    param = ' '.join(sec.get('PARAM', '').split())
    if not steps:
        return None
    block = (f"HEAD: size={(hm.group(1).upper() if hm else '?')}; bg={(bm.group(1) if bm else '?')}; subgrid={(sm.group(1).lower() if sm else '?')}\n"
             f"MECH: {', '.join(sorted(mech))}\nSTEPS:\n" + '\n'.join(f'{i+1}. {s}' for i, s in enumerate(steps)) +
             f"\nPARAM: {param}\nSIZES: {', '.join(f'{a}x{b}' for a, b in sizes) if sizes else '?'}")
    return {'head': {'size': hm.group(1).upper() if hm else None, 'bg': int(bm.group(1)) if bm else None,
                     'subgrid': (sm.group(1).lower() == 'yes') if sm else None},
            'mech': mech, 'steps': steps, 'param': param, 'sizes': sizes, 'block': block}

def head_ok(d, F):
    """L2: HEAD 3주장 vs 기계 사실."""
    h = d['head']
    return (h['size'] == F['size']) and (h['bg'] == F['bg']) and (h['subgrid'] is not None and h['subgrid'] == F['subgrid'])

def sizes_ok(d, t):
    if not d['sizes']:
        return False
    want = [(len(p['output']), len(p['output'][0])) for p in t['train']]
    return len(d['sizes']) >= len(want) and all(d['sizes'][i] == want[i] for i in range(len(want)))

def effective_mechs(d):
    """MECH 줄 ∪ STEPS 본문 EN 검출, LOW_INFO 제외."""
    return (d['mech'] | detect_mechs_en(' '.join(d['steps']))) - LOW_INFO

def literal_flags(d, t):
    """리터럴 파라미터: 절대 좌표 → 즉시 실격. 색 리터럴은 모든 train 쌍 팔레트에 있어야 허용."""
    text = ' '.join(d['steps']) + ' ' + d['param']
    if re.search(r'\b(row|column|col)\s+\d+\b', text, re.I) or re.search(r'\(\s*\d+\s*,\s*\d+\s*\)', text):
        return 'coord'
    pal = None
    for p in t['train']:
        s = set(np.unique(_g(p['input'])).tolist()) | set(np.unique(_g(p['output'])).tolist())
        pal = s if pal is None else (pal & s)
    for c in re.findall(r'colou?r\s*(\d)\b', text, re.I):
        if int(c) not in (pal or set()):
            return 'color'
    return None

def desc_key(d):
    """규칙 동일성 키: STEPS + PARAM (PARAM 이 다르면 다른 규칙 — 회전 각도 등)."""
    s = ' '.join(re.sub(r'[^a-z0-9 ]', ' ', (' '.join(d['steps']) + ' ' + d['param']).lower()).split())
    return hashlib.md5(s.encode()).hexdigest()[:12]

def desc_filter(d, t, F, min_mech=2):
    """반환 (통과여부, 사유). 학습 샘플 인정 전 필터 (왕복 게이트는 별도)."""
    if not head_ok(d, F):
        return False, 'head'
    if not sizes_ok(d, t):
        return False, 'sizes'
    if len(effective_mechs(d)) < min_mech:
        return False, 'mech<2'
    if not (1 <= len(d['mech']) <= 6):
        return False, 'mech_tags'                      # MECH 줄 남발(recall 게임) 차단
    if not d['param']:
        return False, 'param'
    if not (2 <= len(d['steps']) <= 6):
        return False, 'steps'
    lf = literal_flags(d, t)
    if lf:
        return False, 'literal_' + lf
    return True, 'ok'

# ------------------------------------------------------------------------------
# 7. 샌드박스 + 왕복 게이트
# ------------------------------------------------------------------------------
def _alarm(s, f):
    raise TimeoutError()

def extract_code(txt):
    if not txt:
        return None
    txt = re.sub(r'<think>.*?</think>', '', txt, flags=re.S).split('</think>')[-1]
    b = re.findall(r'```python(.*?)```', txt, re.S) or re.findall(r'```(.*?)```', txt, re.S)
    if not b and '```' in txt:                      # 닫는 펜스 없이 잘린 경우
        tail = txt.rsplit('```', 1)[-1]; L = tail.splitlines()
        if L and L[0].strip().lower() in ('python', 'py', ''):
            L = L[1:]
        if not tail.endswith('\n'):                 # 마지막 줄이 잘렸으면 버림
            L = L[:-1]
        if 'def solve' in '\n'.join(L):
            b = ['\n'.join(L)]
    if not b:
        return None
    c = [x for x in b if 'def solve' in x] or b
    return c[-1]

_BLOCKED_BUILTINS = {'__import__', 'open', 'exec', 'eval', 'compile', 'input', 'breakpoint', 'exit', 'quit',
                     'globals', 'locals', 'vars', 'memoryview', 'help', '__loader__', '__spec__'}

def _safe_builtins():
    import builtins
    return {k: getattr(builtins, k) for k in dir(builtins) if k not in _BLOCKED_BUILTINS and not k.startswith('_')}

def _as_grid(out):
    """정수 2D 격자만 인정. 실수(1.9)·bool·object 는 거부 — 묵시적 정수 변환 금지."""
    if out is None:
        return None
    try:
        a = np.asarray(out)
    except Exception:
        return None
    if a.dtype.kind in ('b', 'O', 'U', 'S') or a.ndim != 2 or a.size == 0:
        return None
    if a.dtype.kind == 'f':
        if not np.all(np.isfinite(a)) or not np.all(a == np.round(a)):
            return None
        a = a.astype(int)
    elif a.dtype.kind not in ('i', 'u'):
        return None
    a = a.astype(int)
    if a.shape[0] > 30 or a.shape[1] > 30 or a.min() < 0 or a.max() > 9:
        return None
    return a

def compile_solve(src):
    """★메인 프로세스용은 아니다 — 워커 안에서만 호출. 최상위 코드가 무한루프면 여기서 멈추므로 verify_src 를 쓸 것."""
    if not src or re.search(r'^\s*(import|from)\s+(?!numpy|math|collections|itertools)', src, re.M):
        return None
    ns = {'np': np, 'numpy': np, 'Counter': Counter, 'deque': deque, 'math': math, '__builtins__': _safe_builtins()}
    try:
        exec(compile(src, 'llm', 'exec'), ns)
    except BaseException:
        return None
    fn = ns.get('solve')
    return fn if callable(fn) else None

def run_solve(fn, grid, timeout=EXEC_TIMEOUT):
    signal.signal(signal.SIGALRM, _alarm); signal.alarm(timeout)
    try:
        out = fn([list(map(int, r)) for r in grid]); signal.alarm(0)
        return _as_grid(out)
    except BaseException:
        signal.alarm(0); return None

def _verify_worker(conn, src, train, tests):
    try:
        try:
            import resource
            resource.setrlimit(resource.RLIMIT_AS, (6 << 30, 6 << 30))     # 6GB
        except Exception:
            pass
        signal.signal(signal.SIGALRM, _alarm); signal.alarm(EXEC_TIMEOUT)   # ★최상위 코드(컴파일) 자체에도 시계
        fn = compile_solve(src); signal.alarm(0)
        if fn is None:
            conn.send({'err': 'compile'}); return
        tp = 0
        for p in train:
            o = run_solve(fn, p['input'])
            g = np.asarray(p['output'], dtype=int)
            if o is not None and o.shape == g.shape and np.array_equal(o, g):
                tp += 1
        preds = [(lambda o: o.tolist() if o is not None else None)(run_solve(fn, x['input'])) for x in tests]
        conn.send({'tp': tp, 'preds': preds})
    except BaseException as e:
        try: conn.send({'err': type(e).__name__})
        except Exception: pass

def verify_src(src, t, timeout=None):
    """★프로세스 격리 검증. 반환 {'tp': train통과수, 'te': test정답, 'preds': [test예측], 'err': None|str}.
    자식이 총 시간을 넘기면 강제 종료(err='timeout'). test 정답 비교는 부모가 한다."""
    import multiprocessing as mp
    timeout = timeout or (EXEC_TIMEOUT * (len(t['train']) + len(t['test']) + 1) + 2)
    res = {'tp': 0, 'te': False, 'preds': None, 'err': None}
    if not src:
        res['err'] = 'nocode'; return res
    ctx = mp.get_context('fork')
    parent, child = ctx.Pipe(duplex=False)
    pr = ctx.Process(target=_verify_worker, args=(child, src, t['train'], t['test']), daemon=True)
    pr.start(); child.close()
    msg = None
    try:
        if parent.poll(timeout):
            msg = parent.recv()
    except Exception:
        msg = None
    finally:
        if pr.is_alive():
            pr.kill()
        pr.join(1); parent.close()
    if msg is None:
        res['err'] = 'timeout'; return res
    if 'err' in msg:
        res['err'] = msg['err']; return res
    res['tp'] = msg['tp']; res['preds'] = msg['preds']
    if res['tp'] == len(t['train']) and t.get('_sol') is not None:
        res['te'] = all(p is not None and np.asarray(p).shape == np.asarray(s).shape and np.array_equal(np.asarray(p), np.asarray(s))
                        for p, s in zip(res['preds'], t['_sol'])) and len(res['preds']) == len(t['_sol'])
    return res

def verify_fn(fn, t):
    """(호환) 이미 컴파일된 함수 검증 — 프로세스 격리 없음. 신규 코드는 verify_src 를 쓴다."""
    tp = 0
    for p in t['train']:
        o = run_solve(fn, p['input'])
        if o is not None and o.shape == _g(p['output']).shape and np.array_equal(o, _g(p['output'])):
            tp += 1
    if tp < len(t['train']):
        return tp, False
    te = all((lambda o, g: o is not None and o.shape == g.shape and np.array_equal(o, g))(run_solve(fn, x['input']), _g(t['_sol'][j]))
             for j, x in enumerate(t['test']))
    return tp, bool(te)

NULL_DESC = "(No description is available for this puzzle. Infer the rule from the examples yourself.)"

def roundtrip_batch(engine, items, n_try=3, temp=0.6, max_tokens=1400, lora=False):
    """items = [(task, block), ...] → 한 번의 generate. block=NULL_DESC 이면 무서술 기준선.
    반환 [dict]: ok(test 정답 코드 존재), code, hist(시도별 train통과수, -1=컴파일실패), cands([{src,tp,te,preds}])"""
    if not items:
        return []
    texts = engine.generate([prompt_code(t, b) for t, b in items], think=False, n=n_try, temp=temp, max_tokens=max_tokens, lora=lora)
    out = []
    for (t, b), row in zip(items, texts):
        r = {'ok': False, 'code': None, 'hist': [], 'cands': []}
        for txt in row:
            src = extract_code(txt)
            v = verify_src(src, t)
            tp = -1 if v['err'] else v['tp']
            r['hist'].append(tp)
            r['cands'].append({'src': src, 'tp': v['tp'], 'te': v['te'], 'preds': v['preds'], 'err': v['err']})
            if v['te'] and not r['ok']:
                r['ok'], r['code'] = True, src
        out.append(r)
    return out

def pick_two_slots(cands, n_train):
    """★제출 규칙 (정답 비공개): train 전쌍 통과 후보만 → 같은 test 예측끼리 투표 → 상위 2개 상이 예측. 반환 [preds, preds]."""
    full = [c for c in cands if c.get('tp') == n_train and c.get('preds') and all(p is not None for p in c['preds'])]
    votes = {}
    for c in full:
        k = json.dumps(c['preds'])
        votes.setdefault(k, [0, c['preds']]); votes[k][0] += 1
    ranked = sorted(votes.values(), key=lambda x: -x[0])
    return [p for _, p in ranked[:2]]

# ------------------------------------------------------------------------------
# 8. vLLM 엔진 (교사 채굴 · 왕복 코더 · 평가 공용)
# ------------------------------------------------------------------------------
THINK_CLOSE = ('\n\nConsidering the limited time, I have to give the answer now '
               'based on my reasoning so far.\n</think>\n\n')

class VEngine:
    def __init__(self, model_path, max_model_len=32768, gpu_util=0.92, lora_dir=None, max_lora_rank=64, seed=0):
        from vllm import LLM
        kw = dict(model=model_path, dtype='bfloat16', max_model_len=max_model_len, gpu_memory_utilization=gpu_util,
                  trust_remote_code=True, enable_prefix_caching=True, seed=seed)
        if lora_dir:
            kw.update(enable_lora=True, max_loras=1, max_lora_rank=max_lora_rank)
        try:
            self.llm = LLM(language_model_only=True, **kw)      # VL 클래스에서 텍스트 스택만
        except TypeError:
            self.llm = LLM(**kw)
        self.tok = self.llm.get_tokenizer()
        self.lora_dir = lora_dir
        self._lora = None
        if lora_dir:
            from vllm.lora.request import LoRARequest
            self._lora = LoRARequest('desc', 1, lora_dir)
        self.cut = 0

    def chat_text(self, prompt, think):
        try:
            return self.tok.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False,
                                                add_generation_prompt=True, enable_thinking=bool(think))
        except Exception:
            return self.tok.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)

    def n_tokens(self, prompt, think=False):
        return len(self.tok(self.chat_text(prompt, think), add_special_tokens=False)['input_ids'])

    def _run(self, texts, n, temp, max_tokens, lora, seed, greedy=False, logprobs=None):
        from vllm import SamplingParams
        sp = SamplingParams(n=n, temperature=0.0 if greedy else temp, top_p=1.0 if greedy else 0.95,
                            top_k=-1 if greedy else 20, max_tokens=max_tokens, seed=seed, logprobs=logprobs)
        outs = self.llm.generate(texts, sp, lora_request=(self._lora if lora else None), use_tqdm=False)
        return [[c for c in o.outputs] for o in outs]

    def generate(self, prompts, think=False, n=1, temp=0.8, max_tokens=1024, think_budget=None, answer_tokens=1024,
                 lora=False, seed=None, greedy=False):
        """반환: 프롬프트별 [답 텍스트 × n]. think=True 면 사고 후 답. 사고가 안 닫히면 2패스로 강제 닫고 답."""
        texts = [self.chat_text(p, think) for p in prompts]
        if not think:
            outs = self._run(texts, n, temp, max_tokens, lora, seed, greedy)
            return [[c.text for c in row] for row in outs]
        budget = think_budget or max_tokens
        outs = self._run(texts, n, temp, budget + answer_tokens, lora, seed, greedy)
        res = [[None] * n for _ in prompts]; todo = []
        for i, row in enumerate(outs):
            for j, c in enumerate(row):
                if '</think>' in c.text:
                    res[i][j] = c.text.split('</think>')[-1]
                else:
                    todo.append((i, j, texts[i] + c.text + THINK_CLOSE))
        self.cut = len(todo)
        if todo:
            o2 = self._run([x[2] for x in todo], 1, temp, answer_tokens, lora, seed, greedy)
            for (i, j, _), row in zip(todo, o2):
                res[i][j] = row[0].text.split('</think>')[-1]
        return res

    def greedy_logprob(self, prompt, lora, max_tokens=64):
        """참고용: greedy 텍스트 + 실제 생성 토큰(id 기준)의 로그확률 합. 경로가 다르면 합끼리 비교 불가 → seq_logprob 를 쓴다."""
        outs = self._run([self.chat_text(prompt, False)], 1, 0.0, max_tokens, lora, 0, greedy=True, logprobs=1)
        c = outs[0][0]
        try:
            lp = sum(d[tid].logprob for d, tid in zip(c.logprobs, c.token_ids))
        except Exception:
            lp = float('nan')
        return c.text, lp

    def seq_logprob(self, prompt, completion, lora):
        """★적용 증명용: 고정 완성문(teacher forcing)의 로그확률 합. base 와 lora 를 같은 토큰열에서 비교한다."""
        from vllm import SamplingParams
        head = self.chat_text(prompt, False)
        n_head = len(self.tok(head, add_special_tokens=False)['input_ids'])
        sp = SamplingParams(max_tokens=1, temperature=0.0, prompt_logprobs=1)
        o = self.llm.generate([head + completion], sp, lora_request=(self._lora if lora else None), use_tqdm=False)[0]
        ids, plp = o.prompt_token_ids, o.prompt_logprobs
        tot, n = 0.0, 0
        for i in range(n_head, len(ids)):
            d = plp[i] if i < len(plp) else None
            if d and ids[i] in d:
                tot += d[ids[i]].logprob; n += 1
        return tot, n

# ------------------------------------------------------------------------------
# 9. 어댑터 키 점검 / 리네임 (vLLM 적재 대비)
# ------------------------------------------------------------------------------
def adapter_key_report(adapter_dir):
    from safetensors import safe_open
    f = glob.glob(f'{adapter_dir}/adapter_model.safetensors')
    assert f, f'어댑터 없음: {adapter_dir}'
    with safe_open(f[0], 'pt') as sf:
        keys = list(sf.keys())
    pref = Counter(re.sub(r'layers\.\d+\..*$', 'layers.N', k) for k in keys)
    cfg = json.load(open(f'{adapter_dir}/adapter_config.json'))
    return {'n_keys': len(keys), 'prefixes': dict(pref.most_common(8)), 'r': cfg.get('r'),
            'targets': cfg.get('target_modules') if isinstance(cfg.get('target_modules'), str) else len(cfg.get('target_modules') or []),
            'sample': keys[:3]}

def rename_adapter(adapter_dir, out_dir, old='model.language_model.', new='language_model.model.'):
    """PEFT(HF 5.x) 키 `base_model.model.model.language_model.layers.N` → vLLM 텍스트 스택 이름으로. 원본은 보존."""
    import shutil, torch
    from safetensors.torch import load_file, save_file
    os.makedirs(out_dir, exist_ok=True)
    sd = load_file(f'{adapter_dir}/adapter_model.safetensors')
    new_sd = {k.replace(old, new, 1) if old in k else k: v for k, v in sd.items()}
    save_file(new_sd, f'{out_dir}/adapter_model.safetensors')
    for fn in os.listdir(adapter_dir):
        if fn != 'adapter_model.safetensors' and os.path.isfile(f'{adapter_dir}/{fn}'):
            shutil.copy2(f'{adapter_dir}/{fn}', out_dir)
    cfg = json.load(open(f'{out_dir}/adapter_config.json'))
    tm = cfg.get('target_modules')
    if isinstance(tm, list):                          # 전체 경로 리스트면 같은 치환
        cfg['target_modules'] = [x.replace(old, new, 1) if old in x else x for x in tm]
        json.dump(cfg, open(f'{out_dir}/adapter_config.json', 'w'), indent=1)
    return sum(1 for k in sd if old in k)

# ------------------------------------------------------------------------------
# 10. 데이터셋 유틸
# ------------------------------------------------------------------------------
def read_jsonl(paths, report=True):
    rows, bad, files = [], {}, 0
    for g in paths:
        for f in sorted(glob.glob(g, recursive=True)):
            files += 1
            for ln, line in enumerate(open(f, encoding='utf-8'), 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except Exception:
                    bad[f'{os.path.basename(f)}:{ln}'] = line[:40]
    if report and bad:
        LOG(f'★jsonl 손상 {len(bad)}줄 (파일 {files}개): ' + ', '.join(list(bad)[:5]))
    return rows

def append_jsonl(path, row):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

def balance_weights(rows, source_w=None, clip=(0.5, 4.0)):
    """기전 균등화: 샘플 가중 = 자기 기전들의 (중앙빈도/빈도)^0.5 평균 × 출처 가중, 평균 1 로 정규화."""
    source_w = source_w or {'teacher': 1.0, 'teacher_r1': 1.0, 'hindsight': 1.0, 'human': 3.0}
    freq = Counter()
    for r in rows:
        for m in r.get('mechs', []):
            if m not in LOW_INFO:
                freq[m] += 1
    if not freq:
        return [1.0] * len(rows)
    med = float(np.median(list(freq.values())))
    w = []
    for r in rows:
        ms = [m for m in r.get('mechs', []) if m not in LOW_INFO]
        base = float(np.mean([(med / max(freq[m], 1)) ** 0.5 for m in ms])) if ms else 1.0
        cw = 1.0 if r.get('causal', True) else 0.5          # 무서술 기준선도 푼 태스크 = 서술의 인과 효과 미확인 → 반감
        w.append(min(clip[1], max(clip[0], base)) * source_w.get(r.get('source', 'teacher'), 1.0) * cw)
    mean = float(np.mean(w))
    return [x / mean for x in w]
'''
open('/content/desc_lora_lib.py', 'w', encoding='utf-8').write(LIB_SRC)
shutil.copy2('/content/desc_lora_lib.py', f'{DESC_DIR}/desc_lora_lib.py')
import sys; sys.path.insert(0, '/content')
import importlib, desc_lora_lib as L; importlib.reload(L)
print('desc_lora_lib', L.LIB_VERSION, '/ 기전', len(L.MECHS), '/ 저장', DESC_DIR)


desc_lora_lib 0924b / 기전 40 / 저장 /content/drive/MyDrive/arc_llm/desc_lora


In [3]:
import importlib, transformers.utils.import_utils as iu
importlib.invalidate_caches()
for n in dir(iu):
    f = getattr(iu, n, None)
    if callable(f) and hasattr(f, 'cache_clear'):
        f.cache_clear()
print('torchaudio available:', iu.is_torchaudio_available())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


torchaudio available: True


In [4]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'   # EngineCore 를 같은 프로세스에서 → 진짜 traceback 이 셀에 뜬다
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
3 MiB, 97887 MiB


In [5]:
import contextlib, vllm.utils.system_utils as su
su.suppress_stdout = contextlib.nullcontext          # 코랩 stdout 은 fileno() 가 없다
import vllm.distributed.parallel_state as ps
ps.suppress_stdout = contextlib.nullcontext           # 이미 이름을 가져간 모듈도 갈아끼움

In [6]:
!pip install -q nvidia-cutlass-dsl
import gc, torch, vllm; gc.collect(); torch.cuda.empty_cache()
_orig = vllm.LLM.__init__
def _init(self, *a, **k): k.setdefault('max_num_seqs', 128); _orig(self, *a, **k)
vllm.LLM.__init__ = _init
!nvidia-smi --query-gpu=memory.used --format=csv

memory.used [MiB]
3 MiB


In [7]:
import os, gc, torch
os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
gc.collect(); torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.used --format=csv

memory.used [MiB]
3 MiB


In [8]:
# [셀3] ★교사 채굴 — 27B thinking(예산 14K + 강제종료 2패스) 이 training 태스크를 서술 → 필터 → 왕복 게이트(+무서술 기준선) → desc_teacher_*.jsonl
#   STAGE='gen'. 학생 학습 입력 = prompt_desc(t) (few-shot 없음). 교사 프롬프트에만 형식 few-shot (가상 예 2개 → 검증된 training 서술로 교체).
#   evaluation 은 절대 안 쓴다. 명부는 라운드별: teacher_done_<ROUND>.json (R1 재채굴은 LORA_DIR 을 주면 새 명부).
import os, sys, json, time, random, glob, hashlib
sys.path.insert(0, '/content'); import importlib, desc_lora_lib as L; importlib.reload(L)
from desc_lora_lib import LOG

SMOKE        = True            # ★먼저 True(태스크 4개) → False 본채굴
SMOKE        =  False          # ★먼저 True(태스크 4개) → False 본채굴
BUDGET_MIN   = 20 if SMOKE else 420
N_TASKS      = 4  if SMOKE else 400        # 이번 세션 대상 수 (작은 것부터, 명부 제외)
TASK_BATCH   = 4  if SMOKE else 8          # 한 generate 호출에 묶는 태스크 수
K_DESC       = 4  if SMOKE else 8          # 태스크당 서술 샘플 (thinking)
THINK_BUDGET = 4000 if SMOKE else 14000    # 사고 예산 — 초과 시 강제 닫고 ANSWER_TOK 로 답 (캐글 3072 보다 4~5배)
ANSWER_TOK   = 900
TEMP_DESC    = 0.8
CAP_PER_TASK = 3                            # 태스크당 저장 상한 (기전집합 상이)
PROMPT_TOK_CAP = 6500                       # ★학생 프롬프트(깨끗한 것) 상한 = 학습 MAX_LEN 8192 − 완성 여유
MAX_MODEL_LEN  = 32768
RT_TRY       = 3                            # 왕복: 서술당 코드 시도 수 (무서술 기준선도 같은 수)
LORA_DIR     = None                         # ★R1 재채굴: 어댑터 경로를 주면 lora + no-think 로 서술 (코더는 베이스)
SEED         = 42
random.seed(SEED)

ROUND  = 'r0' if LORA_DIR is None else 'r1_' + hashlib.md5(LORA_DIR.encode()).hexdigest()[:6]
SRC    = 'teacher' if LORA_DIR is None else 'teacher_r1'
TR = L.load_split('training'); TRAIN_TIDS = {tid for tid, _ in TR}
DONE_F = f'{DESC_DIR}/teacher_done_{ROUND}.json'
DONE = json.load(open(DONE_F)) if os.path.exists(DONE_F) else {}
RUN_ID = time.strftime('%m%d_%H%M')
OUT_F  = f'{DESC_DIR}/desc_teacher_{ROUND}_{RUN_ID}.jsonl'
LOG(f'라운드 {ROUND} / 명부 {len(DONE)} (해결 {sum(1 for v in DONE.values() if v.get("n", 0) > 0)}) / lib {L.LIB_VERSION}')

# few-shot 은행: 검증된 training 서술이 있으면 그것 (출처 tid 기록), 없으면 가상 예 2개
_prev = [r for r in L.read_jsonl([f'{DESC_DIR}/desc_*.jsonl']) if r.get('tid') in TRAIN_TIDS and r.get('causal', True)]
FS_BANK = L.dynamic_fewshot_bank(_prev) or L.FEWSHOT_BANK
LOG(f'few-shot 은행 {len(FS_BANK)}개 ({"training 검증분" if _prev else "가상 예"})')

# ensure_model()
# ENG = L.VEngine(LOCAL_DIR, max_model_len=MAX_MODEL_LEN, gpu_util=0.92, seed=SEED, lora_dir=LORA_DIR)
# LOG('엔진 적재 완료' + (f' (+어댑터 {LORA_DIR})' if LORA_DIR else ''))
if 'ENG' in globals() and getattr(ENG, 'lora_dir', None) == LORA_DIR:
    LOG('엔진 재사용 (적재 생략)' + (f' (+어댑터 {LORA_DIR})' if LORA_DIR else ''))
else:
    ensure_model()
    ENG = L.VEngine(LOCAL_DIR, max_model_len=MAX_MODEL_LEN, gpu_util=0.92, seed=SEED, lora_dir=LORA_DIR)
    LOG('엔진 적재 완료' + (f' (+어댑터 {LORA_DIR})' if LORA_DIR else ''))
# 대상: usable + 명부 제외 + 학생 프롬프트 토큰 상한, 작은 것부터
cands = []
for tid, t in TR:
    if tid in DONE or not L.usable(t):
        continue
    F = L.task_facts(t); p_clean = L.prompt_desc(t, F); ntok = ENG.n_tokens(p_clean)
    if ntok > PROMPT_TOK_CAP:
        DONE[tid] = {'n': 0, 'why': f'len{ntok}'}; continue
    cands.append((ntok, tid, t, F, p_clean))
cands.sort(key=lambda x: x[0]); POOL = cands[:N_TASKS]
LOG(f'★대상 {len(POOL)} / 길이초과 제외 {sum(1 for v in DONE.values() if str(v.get("why", "")).startswith("len"))}')
json.dump(DONE, open(DONE_F, 'w'))

DEADLINE = time.monotonic() + BUDGET_MIN * 60
tot = {'round': ROUND, 'lib': L.LIB_VERSION, 'tasks': 0, 'gen': 0, 'parsed': 0, 'filt_ok': 0, 'rt_ok': 0, 'causal': 0, 'saved': 0,
       'cut': 0, 'teacher_prompt_over': 0, 'why': {}, 'gen_sec': 0, 'rt_sec': 0}
for b0 in range(0, len(POOL), TASK_BATCH):
    if time.monotonic() > DEADLINE:
        LOG('예산 마감'); break
    batch = POOL[b0:b0 + TASK_BATCH]
    rng = random.Random(SEED + b0)
    prompts, fs_src = [], []
    for _, _, t, F, _ in batch:
        fs, tids = L.pick_fewshot(set(), k=2, rng=rng, bank=FS_BANK); prompts.append(L.prompt_desc(t, F, fewshot=fs)); fs_src.append(tids)
    # ★실제 호출 길이 검사: 교사 프롬프트 + 사고 + 답이 max_model_len 안에 들어가야 한다
    lim = MAX_MODEL_LEN - THINK_BUDGET - ANSWER_TOK - 64
    over = [i for i, p in enumerate(prompts) if ENG.n_tokens(p, think=True) > lim]
    if over:
        tot['teacher_prompt_over'] += len(over)
        for i in over:
            DONE[batch[i][1]] = {'n': 0, 'why': 'teacher_len'}
        batch = [b for i, b in enumerate(batch) if i not in over]; prompts = [p for i, p in enumerate(prompts) if i not in over]
        fs_src = [x for i, x in enumerate(fs_src) if i not in over]
        if not batch:
            continue
    t0 = time.monotonic()
    if LORA_DIR is None:
        texts = ENG.generate(prompts, think=True, n=K_DESC, temp=TEMP_DESC, think_budget=THINK_BUDGET, answer_tokens=ANSWER_TOK, seed=SEED + b0)
    else:
        texts = ENG.generate([p for _, _, _, _, p in batch], think=False, n=K_DESC * 2, temp=TEMP_DESC, max_tokens=ANSWER_TOK, lora=True, seed=SEED + b0)
    tot['cut'] += ENG.cut; tot['gen_sec'] += time.monotonic() - t0
    LOG(f'배치 {b0//TASK_BATCH}: {len(batch)}태스크 × {K_DESC} 서술 {round(time.monotonic()-t0)}s / 사고 강제종료 {ENG.cut}')
    rt_items, rt_meta, null_items = [], [], []
    for (ntok, tid, t, F, p_clean), row, fsr in zip(batch, texts, fs_src):
        seen, kept, why = set(), [], {}
        for txt in row:
            tot['gen'] += 1
            d = L.parse_desc(txt)
            if d is None:
                why['format'] = why.get('format', 0) + 1; continue
            tot['parsed'] += 1
            ok, r = L.desc_filter(d, t, F)
            if not ok:
                why[r] = why.get(r, 0) + 1; continue
            k = L.desc_key(d)
            if k in seen:
                why['dup'] = why.get('dup', 0) + 1; continue
            seen.add(k); kept.append(d); tot['filt_ok'] += 1
        for k2, v in why.items():
            tot['why'][k2] = tot['why'].get(k2, 0) + v
        for d in kept:
            rt_items.append((t, d['block'])); rt_meta.append((tid, t, F, p_clean, d, ntok, fsr))
        if kept:
            null_items.append((tid, t))
        DONE[tid] = {'n': 0, 'why': json.dumps(why, ensure_ascii=False)[:80]}
    if rt_items:
        t0 = time.monotonic()
        res = L.roundtrip_batch(ENG, rt_items + [(t, L.NULL_DESC) for _, t in null_items], n_try=RT_TRY)
        tot['rt_sec'] += time.monotonic() - t0
        null_ok = {tid: r['ok'] for (tid, _), r in zip(null_items, res[len(rt_items):])}
        null_hist = {tid: r['hist'] for (tid, _), r in zip(null_items, res[len(rt_items):])}
        res = res[:len(rt_items)]
        LOG(f'   왕복 {len(rt_items)}서술 + 무서술 {len(null_items)} {round(time.monotonic()-t0)}s → 통과 {sum(1 for r in res if r["ok"])} / 무서술 통과 {sum(null_ok.values())}')
        per_tid = {}
        for (tid, t, F, p_clean, d, ntok, fsr), r in zip(rt_meta, res):
            if not r['ok']:
                continue
            tot['rt_ok'] += 1
            lst = per_tid.setdefault(tid, []); mech = frozenset(L.effective_mechs(d))
            if len(lst) >= CAP_PER_TASK or any(mech == m for _, m in lst):
                continue
            lst.append((d, mech)); causal = not null_ok.get(tid, False); tot['causal'] += causal
            L.append_jsonl(OUT_F, {'tid': tid, 'source': SRC, 'round': ROUND, 'prompt': p_clean, 'completion': d['block'],
                                   'mechs': sorted(mech), 'code': r['code'], 'rt_hist': r['hist'], 'null_hist': null_hist.get(tid),
                                   'causal': causal, 'fewshot_src': fsr, 'prompt_tok': ntok, 'lib': L.LIB_VERSION, 'run': RUN_ID})
            tot['saved'] += 1; DONE[tid] = {'n': len(lst), 'causal': causal}
        for tid, lst in per_tid.items():
            LOG(f'   ★[{tid}] 저장 {len(lst)}{"" if not null_ok.get(tid) else " (무서술도 통과 → 비인과, 가중 반감)"} :: ' + ' | '.join(sorted(lst[0][1])))
    tot['tasks'] += len(batch)
    json.dump(DONE, open(DONE_F, 'w'))
    json.dump(tot, open(f'{DESC_DIR}/teacher_stats_{ROUND}_{RUN_ID}.json', 'w'), ensure_ascii=False, indent=1)

print('=' * 76); print(json.dumps(tot, ensure_ascii=False, indent=1))
print(f"태스크 {tot['tasks']} → 서술 {tot['gen']} → 형식 {tot['parsed']} → 필터 {tot['filt_ok']} → 왕복 {tot['rt_ok']} → 저장 {tot['saved']} (인과 {tot['causal']})")
print('★산출:', OUT_F)
if SMOKE:
    print('  → SMOKE=False 로 본채굴. 세션이 끊기면 셀1→셀2→셀3 재실행이 이 라운드 명부부터 이어달린다.')


[11:10:43] 라운드 r0 / 명부 95 (해결 2) / lib 0924b
[11:10:43] few-shot 은행 1개 (training 검증분)
INFO 09-24 11:10:43 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 32768, 'enable_prefix_caching': True, 'max_num_seqs': 128, 'disable_log_stats': True, 'language_model_only': True, 'model': '/content/q38_bf16'}
WARNING 09-24 11:10:43 [arg_utils.py:1812] The global random seed is set to 42. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-24 11:10:43 [model.py:692] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 09-24 11:10:43 [model.py:2030] Using max model len 32768
WARNING 09-24 11:10:43 [model.py:994] Model does not support mm_device_do_normalize, forcing mm_device_do_normalize = False.
INFO 09-24 11:10:44 [registry.py:146] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
INFO 09-24

INFO 09-24 11:10:48 [cuda.py:538] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 09-24 11:10:48 [flash_attn.py:1116] Using FlashAttention version 2
INFO 09-24 11:10:48 [weight_utils.py:895] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 51.75 GiB. Available RAM: 170.92 GiB.
INFO 09-24 11:10:48 [weight_utils.py:918] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/18 [00:00<?, ?it/s]


INFO 09-24 11:10:53 [default_loader.py:430] Loading weights took 4.48 seconds
INFO 09-24 11:10:54 [model_runner.py:428] Model loading took 50.22 GiB memory and 6.825406 seconds
INFO 09-24 11:10:54 [topk_topp_sampler.py:62] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
INFO 09-24 11:10:54 [interface.py:933] Setting attention block size to 784 tokens to ensure that attention page size is >= mamba page size.
INFO 09-24 11:10:54 [interface.py:957] Padding mamba page size by 0.13% to ensure that mamba page size and attention page size are exactly equal.
INFO 09-24 11:10:54 [utils.py:320] Using LBNHC KV cache layout.
INFO 09-24 11:10:55 [caching.py:343] reconstructed serializable fn from standalone compile artifacts. num_artifacts=20 num_submods=65
INFO 09-24 11:10:55 [decorators.py:313] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/44f0b6dc40659cbbd970481b0ed0015685b76bc306bd5cf66672a95886b47eb8/rank_0_0/model
I

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 16.73it/s]


...

2026-09-24 11:11:18,008 - INFO - autotuner.py:2789 - flashinfer.jit: [Autotuner]: Loaded 0 configs from /root/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.18.post1/7e51d853b8fc04c1cd1209ffebeea35e1a15b9370135eef461479cab9f766fd2/autotune_configs.json
2026-09-24 11:11:18,009 - INFO - autotuner.py:972 - flashinfer.jit: [Autotuner]: Autotuning process starts ...


INFO 09-24 11:11:18 [kernel_warmup.py:359] Running FlashInfer autotune with 16384 tokens and token buckets (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 768, 1024, 1280, 1536, 1792, 2048, 2560, 3072, 3584, 4096, 8192, 16384).


2026-09-24 11:11:18,026 - INFO - autotuner.py:995 - flashinfer.jit: [Autotuner]: Autotuning process ends
2026-09-24 11:11:18,029 - INFO - autotuner.py:2652 - flashinfer.jit: [Autotuner]: Saved 0 configs to /root/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.18.post1/7e51d853b8fc04c1cd1209ffebeea35e1a15b9370135eef461479cab9f766fd2/autotune_configs.json (0 new, 0 from previous config)
Capturing CUDA graphs (FULL): 100%|██████████| 19/19 [00:01<00:00, 17.52it/s]


INFO 09-24 11:11:28 [model_runner.py:1066] Graph capturing finished in 8 secs, took 0.20 GiB
INFO 09-24 11:11:28 [gpu_worker.py:825] CUDA graph pool memory: 0.2 GiB (actual), 0.54 GiB (estimated), difference: 0.34 GiB (170.6%).
INFO 09-24 11:11:28 [gpu_worker.py:888] Free memory on device (94.43/94.97 GiB) on startup. Desired GPU memory utilization is (0.92, 87.37 GiB). Actual usage is 50.61 GiB for consumed memory (weights + non-torch), 2.82 GiB for peak activation, and 0.2 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=36071071622` (33.59 GiB) to fit into requested memory, or `--kv-cache-memory=43643696128` (40.65 GiB) to fully utilize gpu memory. Current kv cache memory in use is 33.94 GiB.
INFO 09-24 11:11:29 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
INFO 09-24 11:11:29 [torch_utils.py:287] Reducing Torch threads from 24 to 1 for serving. Set OMP_NUM_THREADS in the exter

KeyboardInterrupt: 

In [9]:
import re
PAT = re.compile(r"training|evaluation|대상|few.?shot|challenges|solutions|명부|TARGET|SPLIT|SUBSET", re.I)
for ci, src in enumerate(In):
    if not src or "PAT = re.compile" in src: continue
    lines = src.splitlines()
    hits = [n for n, l in enumerate(lines) if PAT.search(l)]
    if not hits: continue
    print(f"\n===== In[{ci}] =====")
    shown = set()
    for n in hits:
        for k in range(max(0, n-1), min(len(lines), n+2)):
            if k not in shown:
                print(f"{k+1:4d}: {lines[k][:160]}"); shown.add(k)


===== In[1] =====
  23: os.makedirs(DESC_DIR, exist_ok=True)
  24: if not os.path.exists('/content/ARC-AGI-2/data/training'):
  25:     subprocess.run('git clone -q --depth 1 https://github.com/arcprize/ARC-AGI-2 /content/ARC-AGI-2', shell=True, check=True)

===== In[2] =====
  16:   * 샌드박스 + 왕복 게이트 (서술 → 코드 → train 전쌍 + test)
  17:   * 사람 서술(eval119 / training memo) 파서
  18:   * vLLM 엔진 래퍼 (thinking 2패스, LoRARequest)
  37: # ------------------------------------------------------------------------------
  38: def load_split(split, root=None):
  39:     root = root or ARC_ROOT
  40:     fs = sorted(glob.glob(f'{root}/data/{split}/*.json'))
  41:     out = []
 233: # ------------------------------------------------------------------------------
 234: # 4. 사람 서술 파서 (eval119 메모 / training 메모)
 235: # ------------------------------------------------------------------------------
 238:     D, cur = {}, None
 239:     for raw in text.replace('\r', '').split('\n'):
 240:         l = raw.strip

In [ ]:
# [셀4] ★사후 서술(hindsight) — 이미 풀린 training 태스크의 정답 코드 → 교사가 서술 → 왕복 게이트(코드 비공개) → desc_hind_*.jsonl
#   STAGE='gen', 셀3 과 같은 런타임에서 이어서 실행 가능 (ENG 재사용).
#   코드 원천 (있는 것만 쓴다 — ★셀3 산출은 읽지 않는다. 셀3은 이 셀과 독립):
#     ① MyDrive/arc_llm/**/SOLVED_tr_<tid>.py          (이전 채굴 노트북 산출, '# RULE: ...' 헤더 + 코드)
#     ② MyDrive/arc_llm/**/dataset_code_*.jsonl          (이전 채굴: {tid, completion: ```python ...```})
#     ③ DESC_DIR/solved_codes.jsonl                      ({tid, code}) — 규칙함수·렌즈 해결분을 여기에 덤프해 두면 된다
#   각 코드는 여기서 다시 실행 검증(프로세스 격리)한 것만 쓴다.
import os, sys, json, time, random, glob, re
sys.path.insert(0, '/content'); import importlib, desc_lora_lib as L; importlib.reload(L)
from desc_lora_lib import LOG

SMOKE      = True
BUDGET_MIN = 15 if SMOKE else 240
N_TASKS    = 4  if SMOKE else 600
TASK_BATCH = 8
K_DESC     = 4                  # 코드를 보고 쓰므로 no-think, 4개
TEMP_DESC  = 0.8
CAP_PER_TASK = 3
PROMPT_TOK_CAP = 6500
SEED = 43

TR = dict(L.load_split('training'))
DONE_F = f'{DESC_DIR}/hind_done.json'
DONE = json.load(open(DONE_F)) if os.path.exists(DONE_F) else {}
RUN_ID = time.strftime('%m%d_%H%M')
OUT_F = f'{DESC_DIR}/desc_hind_{RUN_ID}.jsonl'

# ---- 코드 수집 + 실제 검증 (train 전쌍 + test 통과하는 코드만) ----
codes = {}
for f in glob.glob(f'{DRIVE_ROOT}/**/SOLVED_tr_*.py', recursive=True):
    tid = os.path.basename(f)[10:18]
    src = open(f, encoding='utf-8').read()
    src = '\n'.join(l for l in src.split('\n') if not l.startswith('# RULE:'))
    codes.setdefault(tid, []).append(src)
for r in L.read_jsonl([f'{DRIVE_ROOT}/**/dataset_code_*.jsonl']):
    src = L.extract_code(r.get('completion', ''))
    if src and r.get('tid'):
        codes.setdefault(r['tid'], []).append(src)
for r in L.read_jsonl([f'{DESC_DIR}/solved_codes.jsonl']):
    if r.get('tid') and r.get('code'):
        codes.setdefault(r['tid'], []).append(r['code'])
good = {}
for tid, srcs in codes.items():
    t = TR.get(tid)
    if t is None or tid in DONE or not L.usable(t):
        continue
    for src in srcs:
        if L.verify_src(src, t)['te']:
            good[tid] = src; break
LOG(f'코드 원천 {len(codes)} tid → 검증 통과 {len(good)} (명부 제외 후)')
assert good, '★사용할 정답 코드가 없다 — 위 ①②③ 경로 중 하나를 채울 것'

if 'ENG' not in globals():
    ensure_model(); ENG = L.VEngine(LOCAL_DIR, max_model_len=32768, gpu_util=0.92, seed=SEED)

items = []
for tid, src in good.items():
    t = TR[tid]; F = L.task_facts(t); p_clean = L.prompt_desc(t, F)
    ntok = ENG.n_tokens(p_clean)
    if ntok > PROMPT_TOK_CAP:
        DONE[tid] = {'n': 0, 'why': f'len{ntok}'}; continue
    items.append((ntok, tid, t, F, p_clean, src))
items.sort(key=lambda x: x[0]); POOL = items[:N_TASKS]
LOG(f'★대상 {len(POOL)}')

DEADLINE = time.monotonic() + BUDGET_MIN * 60
tot = {'tasks': 0, 'gen': 0, 'parsed': 0, 'filt_ok': 0, 'rt_ok': 0, 'saved': 0, 'why': {}}
for b0 in range(0, len(POOL), TASK_BATCH):
    if time.monotonic() > DEADLINE:
        LOG('예산 마감'); break
    batch = POOL[b0:b0 + TASK_BATCH]
    rng = random.Random(SEED + b0)
    prompts = [L.prompt_desc_from_code(t, src, F, fewshot=L.pick_fewshot(set(), k=2, rng=rng)[0]) for _, _, t, F, _, src in batch]
    lim = 32768 - 900 - 64
    keep = [i for i, p in enumerate(prompts) if ENG.n_tokens(p) <= lim]        # ★실제 호출 길이(코드 포함) 검사
    if len(keep) < len(batch):
        for i in range(len(batch)):
            if i not in keep: DONE[batch[i][1]] = {'n': 0, 'why': 'teacher_len'}
        batch = [batch[i] for i in keep]; prompts = [prompts[i] for i in keep]
        if not batch: continue
    texts = ENG.generate(prompts, think=False, n=K_DESC, temp=TEMP_DESC, max_tokens=900, seed=SEED + b0)
    rt_items, rt_meta = [], []
    for (ntok, tid, t, F, p_clean, src), row in zip(batch, texts):
        seen, why = set(), {}
        for txt in row:
            tot['gen'] += 1
            d = L.parse_desc(txt)
            if d is None:
                why['format'] = why.get('format', 0) + 1; continue
            tot['parsed'] += 1
            ok, r = L.desc_filter(d, t, F)
            if not ok:
                why[r] = why.get(r, 0) + 1; continue
            k = L.desc_key(d)
            if k in seen:
                continue
            seen.add(k); tot['filt_ok'] += 1
            rt_items.append((t, d['block'])); rt_meta.append((tid, t, p_clean, d, ntok))
        for k2, v in why.items():
            tot['why'][k2] = tot['why'].get(k2, 0) + v
        DONE[tid] = {'n': 0}
    null_tids = sorted({m[0] for m in rt_meta}); tid2t = {m[0]: m[1] for m in rt_meta}
    res = L.roundtrip_batch(ENG, rt_items + [(tid2t[x], L.NULL_DESC) for x in null_tids], n_try=3)   # ★코더는 원본 코드를 보지 않는다 + 무서술 기준선
    null_ok = {x: r['ok'] for x, r in zip(null_tids, res[len(rt_items):])}; null_hist = {x: r['hist'] for x, r in zip(null_tids, res[len(rt_items):])}
    res = res[:len(rt_items)]
    per_tid = {}
    for (tid, t, p_clean, d, ntok), r in zip(rt_meta, res):
        if not r['ok']:
            continue
        tot['rt_ok'] += 1
        lst = per_tid.setdefault(tid, []); mech = frozenset(L.effective_mechs(d))
        if len(lst) >= CAP_PER_TASK or any(mech == m for _, m in lst):
            continue
        lst.append((d, mech)); causal = not null_ok.get(tid, False)
        L.append_jsonl(OUT_F, {'tid': tid, 'source': 'hindsight', 'prompt': p_clean, 'completion': d['block'],
                               'mechs': sorted(mech), 'code': r['code'], 'rt_hist': r['hist'], 'null_hist': null_hist.get(tid), 'causal': causal,
                               'prompt_tok': ntok, 'lib': L.LIB_VERSION, 'run': RUN_ID})
        tot['saved'] += 1; DONE[tid] = {'n': len(lst), 'causal': causal}
    tot['tasks'] += len(batch)
    LOG(f"배치 {b0//TASK_BATCH}: 서술 {len(rt_items)} → 왕복통과 {sum(1 for r in res if r['ok'])} / 무서술 통과 {sum(null_ok.values())} → 저장 {sum(len(v) for v in per_tid.values())}")
    json.dump(DONE, open(DONE_F, 'w')); json.dump(tot, open(f'{DESC_DIR}/hind_stats_{RUN_ID}.json', 'w'), ensure_ascii=False, indent=1)

print('=' * 76); print(json.dumps(tot, ensure_ascii=False, indent=1)); print('★산출:', OUT_F)


In [ ]:
# [셀5] ★사람 서술(training) → 형식 서술 → 왕복 게이트 → desc_human_*.jsonl   (source='human', 학습 가중 3배)
#   STAGE='gen'. 입력: DESC_DIR/human_training_memo.txt — eval119 메모와 같은 문법인데 번호 대신 tid 줄:
#       00576224
#       크기 다름
#       일치 x
#       ////
#       타일 반복. 확대 3배
#       ###############
#       007bbfb7
#       ...
#   대상 태스크 고르기: 아래 PICK 블록이 "실패 축" 기전(조립·개수·제어표·홀)을 가진 training 태스크 20개를 추천해 출력한다.
import os, sys, json, time, random
sys.path.insert(0, '/content'); import importlib, desc_lora_lib as L; importlib.reload(L)
from desc_lora_lib import LOG

MEMO_F = f'{DESC_DIR}/human_training_memo.txt'
K_DESC, TEMP_DESC, SEED = 4, 0.7, 44
TR = dict(L.load_split('training'))

# ---- PICK: 아직 서술이 없는 training 태스크 중, 교사가 못 풀었고(teacher_done n=0) 크기·홀·분단선 사실로 실패축 후보를 추천 ----
tdone = json.load(open(f'{DESC_DIR}/teacher_done.json')) if os.path.exists(f'{DESC_DIR}/teacher_done.json') else {}
rec = []
for tid, t in TR.items():
    if not L.usable(t) or tdone.get(tid, {}).get('n', 0) > 0:
        continue
    F = L.task_facts(t)
    holes = sum(d['holes_in'] for d in F['per']); seps = sum(d['sep_in'] for d in F['per'])
    nobj = sum(len(d['obj_in']) for d in F['per'])
    score = (F['size'] == 'DIFFERENT') * 2 + (holes > 0) * 2 + (seps > 0) + (nobj >= 6)
    rec.append((score, L.task_cells(t), tid))
rec.sort(key=lambda x: (-x[0], x[1]))
print('★사람 서술 추천 training 20 (실패축 후보, 작은 것부터):', ' '.join(tid for _, _, tid in rec[:20]))

if not os.path.exists(MEMO_F):
    print(f'★메모 파일 없음: {MEMO_F} — 위 tid 로 메모를 쓴 뒤 이 셀을 다시 실행'); raise SystemExit
D = L.parse_human_memo(open(MEMO_F, encoding='utf-8', errors='ignore').read(), key_re=r'^([0-9a-f]{8})$')
D = {k: v for k, v in D.items() if k in TR and (v['head'] or v['rule'])}
LOG(f'사람 서술 {len(D)}개')
if 'ENG' not in globals():
    ensure_model(); ENG = L.VEngine(LOCAL_DIR, max_model_len=32768, gpu_util=0.92, seed=SEED)
RUN_ID = time.strftime('%m%d_%H%M'); OUT_F = f'{DESC_DIR}/desc_human_{RUN_ID}.jsonl'

meta, prompts = [], []
for tid, d in D.items():
    t = TR[tid]; F = L.task_facts(t); p_clean = L.prompt_desc(t, F)
    meta.append((tid, t, F, p_clean, d)); prompts.append(L.prompt_desc_from_memo(t, L.human_memo_text(d), F))
texts = ENG.generate(prompts, think=False, n=K_DESC, temp=TEMP_DESC, max_tokens=900, seed=SEED)
rt_items, rt_meta = [], []
for (tid, t, F, p_clean, d0), row in zip(meta, texts):
    seen = set()
    for txt in row:
        d = L.parse_desc(txt)
        if d is None:
            continue
        ok, r = L.desc_filter(d, t, F)
        if not ok:
            continue
        # 사람 기전이 보존됐는지: 메모 기전집합의 60% 이상이 서술에 나와야 한다
        hm = L.human_mech_set(d0); em = L.effective_mechs(d)
        if hm and len(hm & em) < 0.6 * len(hm):
            continue
        k = L.desc_key(d)
        if k in seen:
            continue
        seen.add(k); rt_items.append((t, d['block'])); rt_meta.append((tid, p_clean, d))
null_tids = sorted({m[0] for m in rt_meta})
res = L.roundtrip_batch(ENG, rt_items + [(TR[x], L.NULL_DESC) for x in null_tids], n_try=4)
null_ok = {x: r['ok'] for x, r in zip(null_tids, res[len(rt_items):])}; res = res[:len(rt_items)]
saved = {}
for (tid, p_clean, d), r in zip(rt_meta, res):
    if not r['ok'] or saved.get(tid, 0) >= 2:
        continue
    L.append_jsonl(OUT_F, {'tid': tid, 'source': 'human', 'prompt': p_clean, 'completion': d['block'],
                           'mechs': sorted(L.effective_mechs(d)), 'code': r['code'], 'rt_hist': r['hist'], 'causal': not null_ok.get(tid, False),
                           'lib': L.LIB_VERSION, 'run': RUN_ID})
    saved[tid] = saved.get(tid, 0) + 1
print('=' * 76)
print(f'사람 서술 {len(D)} → 형식화 {len(rt_items)} → 왕복 통과 저장 {sum(saved.values())} (태스크 {len(saved)})')
miss = [tid for tid in D if tid not in saved]
print('왕복 실패 태스크(메모가 코드로 안 옮겨짐 — 단계가 모호한지 확인):', ' '.join(miss))
print('★산출:', OUT_F)


In [ ]:
# [셀6] 데이터셋 조립 — desc_teacher/hind/human_*.jsonl → 출처·형식·길이 검증 → 동일규칙 그룹에서 출처 우선(human>teacher>hindsight)
#   → 태스크당 상한 3(기전집합 상이) → tid 분할(고정 파일) → 기전균등화 가중(train 에서만 계산) → desc_sft_{train,val}.jsonl
#   STAGE 무관 (토크나이저만 필요). GPU 불필요.
import os, sys, json, glob, random, hashlib
from collections import Counter
import numpy as np
sys.path.insert(0, '/content'); import importlib, desc_lora_lib as L; importlib.reload(L)
from desc_lora_lib import LOG

MAX_LEN, VAL_FRAC, PER_TID_CAP, SEED = 8192, 0.05, 3, 42
SOURCE_W = {'teacher': 1.0, 'teacher_r1': 1.0, 'hindsight': 1.0, 'human': 3.0}
PRI = {'human': 0, 'teacher': 1, 'teacher_r1': 1, 'hindsight': 2}
VAL_F = f'{DESC_DIR}/val_tids.json'                       # ★고정 validation tid (한 번 정하면 유지)
random.seed(SEED)

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(LOCAL_DIR if os.path.exists(f'{LOCAL_DIR}/config.json') else MODEL_ID)
def chat_prompt(p):
    try:
        return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)

TRAIN = dict(L.load_split('training')); TRAIN_TIDS = set(TRAIN)
EVAL_TIDS = {tid for tid, _ in L.load_split('evaluation')}
rows = L.read_jsonl([f'{DESC_DIR}/desc_teacher_*.jsonl', f'{DESC_DIR}/desc_hind_*.jsonl', f'{DESC_DIR}/desc_human_*.jsonl'])
LOG(f'원시 {len(rows)}')
drop = Counter()
# 1) 출처·형식·프롬프트 일치·길이 검증 (상한 적용 전에)
ok_rows = []
for r in rows:
    tid = r.get('tid')
    if tid in EVAL_TIDS:
        drop['eval_leak'] += 1; continue
    if tid not in TRAIN_TIDS:
        drop['not_training'] += 1; continue
    if r.get('lib') not in (None, L.LIB_VERSION):
        drop['lib_mismatch'] += 1; continue           # 다른 프롬프트 버전으로 채굴된 행
    d = L.parse_desc(r.get('completion', ''))
    if d is None:
        drop['format'] += 1; continue
    F = L.task_facts(TRAIN[tid])
    if r.get('prompt') != L.prompt_desc(TRAIN[tid], F):
        drop['prompt_mismatch'] += 1; continue        # 학습 입력은 현재 lib 의 prompt_desc 와 글자 단위로 같아야 한다
    okf, why = L.desc_filter(d, TRAIN[tid], F)
    if not okf:
        drop['filter_' + why] += 1; continue
    n = len(tok(chat_prompt(r['prompt']), add_special_tokens=False)['input_ids']) + len(tok(d['block'], add_special_tokens=False)['input_ids']) + 4
    if n > MAX_LEN:
        drop['len'] += 1; continue
    r['completion'] = d['block']; r['ntok'] = n; r['mechs'] = sorted(L.effective_mechs(d)); r['_mset'] = frozenset(r['mechs'])
    r['_key'] = L.desc_key(d); r.setdefault('causal', True); r.setdefault('source', 'teacher')
    ok_rows.append(r)
# 2) 동일 규칙 그룹(tid, key) → 출처 우선순위 최상 1개 (사람 서술이 교사와 같아도 사람 것이 남는다)
grp = {}
for r in ok_rows:
    k = (r['tid'], r['_key'])
    if k not in grp or PRI.get(r['source'], 9) < PRI.get(grp[k]['source'], 9):
        grp[k] = r
drop['dup_rule'] = len(ok_rows) - len(grp)
# 3) 태스크당 상한 — 출처 우선, 기전집합 상이 우선
by = {}
for r in grp.values():
    by.setdefault(r['tid'], []).append(r)
kept = []
for tid, lst in by.items():
    lst.sort(key=lambda r: (PRI.get(r['source'], 9), not r['causal']))
    got = [r for r in lst[:0]]
    for r in lst:
        if len(got) < PER_TID_CAP and all(r['_mset'] != g['_mset'] for g in got):
            got.append(r)
    for r in lst:
        if len(got) < PER_TID_CAP and r not in got:
            got.append(r)
    kept += got
drop['cap'] = len(grp) - len(kept)
assert kept, '★남은 샘플 0 — drop 내역 확인: ' + json.dumps(dict(drop), ensure_ascii=False)
# 4) tid 분할 (고정 파일)
tids = sorted({r['tid'] for r in kept})
if os.path.exists(VAL_F):
    val_t = set(json.load(open(VAL_F))) & set(tids)
    new = [t for t in tids if t not in val_t]; random.shuffle(new)
    need = max(1, int(len(tids) * VAL_FRAC)) - len(val_t)
    val_t |= set(new[:max(0, need)])
else:
    random.shuffle(tids); val_t = set(tids[:max(1, int(len(tids) * VAL_FRAC))])
json.dump(sorted(val_t), open(VAL_F, 'w'))
train = [r for r in kept if r['tid'] not in val_t]; val = [r for r in kept if r['tid'] in val_t]
assert train and val, f'★train {len(train)} / val {len(val)} — 비어 있음'
# 5) 가중 (train 에서만 계산)
for r, x in zip(train, L.balance_weights(train, SOURCE_W)):
    r['weight'] = round(float(x), 3)
for r in val:
    r['weight'] = 1.0
for r in kept:
    r.pop('_mset', None); r.pop('_key', None)
with open(f'{DESC_DIR}/desc_sft_train.jsonl', 'w', encoding='utf-8') as f:
    for r in train: f.write(json.dumps(r, ensure_ascii=False) + '\n')
with open(f'{DESC_DIR}/desc_sft_val.jsonl', 'w', encoding='utf-8') as f:
    for r in val: f.write(json.dumps(r, ensure_ascii=False) + '\n')
lens = [r['ntok'] for r in kept]
mf = Counter(m for r in train for m in r['mechs'] if m not in L.LOW_INFO); src = Counter(r['source'] for r in kept)
stats = {'lib': L.LIB_VERSION, 'raw': len(rows), 'drop': dict(drop), 'kept': len(kept), 'train': len(train), 'val': len(val), 'tasks': len(tids),
         'sources': dict(src), 'causal_rate': round(sum(r['causal'] for r in kept) / len(kept), 3),
         'len_median': int(np.median(lens)), 'len_max': max(lens), 'mech_freq': dict(mf.most_common()),
         'weight_by_source': {s_: round(float(np.mean([r['weight'] for r in train if r['source'] == s_])), 2) for s_ in src if any(r['source'] == s_ for r in train)}}
json.dump(stats, open(f'{DESC_DIR}/desc_sft_stats.json', 'w'), ensure_ascii=False, indent=1)
print('=' * 76); print(json.dumps({k: v for k, v in stats.items() if k != 'mech_freq'}, ensure_ascii=False, indent=1))
print('기전 빈도 (train, LOW_INFO 제외):'); print('  ' + ' '.join(f'{m}:{n}' for m, n in mf.most_common()))
thin = [m for m in L.MECH_KEYS if m not in L.LOW_INFO and mf.get(m, 0) < 15]
print(f'★15개 미만 기전 {len(thin)}: {thin}  → 사람 서술(셀5) 대상으로 우선')


In [ ]:
# [셀7] ★QLoRA 학습 — desc_sft_train.jsonl → 서술 어댑터 (STAGE='train', 새 런타임: 셀1 → 셀2 → 셀7)
#   기존 코랩_전체_채굴_훈련_판정 셀3 골격(nf4·청크 CE·PagedAdamW8bit·ckpt 이어달림) + ★샘플 가중 손실 + r64 + 키 점검/리네임 + PEFT 기준 출력
import os, sys, json, glob, time, random, math, gc, shutil
sys.path.insert(0, '/content'); import importlib, desc_lora_lib as L; importlib.reload(L)
from desc_lora_lib import LOG
import numpy as np, torch

class A:
    smoke  = False        # True: 16샘플·20스텝
    resume = False
    epochs = 2
    lr     = 1e-4
    r      = 64           # ★어휘·문체까지 옮기므로 r32 보다 위
    max_len = 8192
GRAD_ACC, LORA_DROP, CE_CHUNK, SAVE_EVERY, LOG_EVERY, SEED = 8, 0.05, 1024, 20, 5, 42
LORA_ALPHA = 2 * A.r
OUT_DIR = f'{DRIVE_ROOT}/lora_desc_q38'
random.seed(SEED)
assert torch.cuda.is_available()
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
MAX_LEN = A.max_len if VRAM >= 70 else 4096
LOG(f'GPU {torch.cuda.get_device_name(0)} {VRAM:.0f}GB → MAX_LEN {MAX_LEN}')

train_rows = L.read_jsonl([f'{DESC_DIR}/desc_sft_train.jsonl']); val_rows = L.read_jsonl([f'{DESC_DIR}/desc_sft_val.jsonl'])
assert len(train_rows) >= (4 if A.smoke else 30), f'★샘플 부족 {len(train_rows)} — 셀6 먼저'
if A.smoke:
    train_rows = train_rows[:16]; val_rows = val_rows[:4]
LOG(f'train {len(train_rows)} / val {len(val_rows)} / epochs {A.epochs}')

ensure_model()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
tok = AutoTokenizer.from_pretrained(LOCAL_DIR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
def chat_prompt(p):
    try:
        return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
END = '<|im_end|>' if '<|im_end|>' in tok.get_vocab() else tok.eos_token
def encode(r):
    ip = tok(chat_prompt(r['prompt']), add_special_tokens=False)['input_ids']
    ic = tok(r['completion'].strip() + END, add_special_tokens=False)['input_ids']
    if len(ip) + len(ic) > MAX_LEN:
        return None
    return {'input_ids': ip + ic, 'labels': [-100] * len(ip) + ic, 'w': float(r.get('weight', 1.0))}
def encode_all(rs, name):
    out, skip = [], 0
    for r in rs:
        e = encode(r)
        if e is None: skip += 1
        else: out.append(e)
    LOG(f'{name}: {len(out)} 인코딩 (길이초과 {skip})'); return out
train_ds, val_ds = encode_all(train_rows, 'train'), encode_all(val_rows, 'val')
_lens = sorted(len(e['input_ids']) for e in train_ds); LOG(f'토큰 길이 중앙값 {_lens[len(_lens)//2]} / 최대 {_lens[-1]}')
LOG('라벨 구간 미리보기:', repr(tok.decode([t for t in train_ds[0]['labels'] if t != -100])[:60]))
assert tok.decode([t for t in train_ds[0]['labels'] if t != -100]).lstrip().startswith('HEAD:'), '★라벨이 HEAD 로 시작하지 않음 — 템플릿 확인'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
                         llm_int8_skip_modules=['lm_head', 'visual', 'vision_tower', 'vision_model', 'multi_modal_projector', 'merger'])
LOG('4bit 적재...'); t = time.monotonic()
model = AutoModelForCausalLM.from_pretrained(LOCAL_DIR, quantization_config=bnb, dtype=torch.bfloat16, device_map={'': 0}, low_cpu_mem_usage=True)
LOG(f'적재 {(time.monotonic()-t)/60:.1f}분 / {torch.cuda.memory_allocated()/1e9:.1f}GB')
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False})
PROJ = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')     # ★GDN in_proj_* 는 절대 타겟 안 함
targets = [n for n, m in model.named_modules() if 'Linear4bit' in type(m).__name__ and n.endswith(PROJ) and not any(k in n for k in ('visual', 'vision'))]
if any('language_model' in n for n in targets):
    targets = [n for n in targets if 'language_model' in n]
assert not any('in_proj' in n or 'linear_attn' in n for n in targets), '★GDN 모듈이 타겟에 들어감'
LOG(f'LoRA 타깃 {len(targets)}개 (예: {targets[0]})')

CKPT = f'{OUT_DIR}/ckpt'; state = {'step': 0, 'epoch': 0, 'cursor': 0}
if A.resume and os.path.exists(f'{CKPT}/adapter_config.json'):
    model = PeftModel.from_pretrained(model, CKPT, is_trainable=True)
    try: state = json.load(open(f'{CKPT}/state.json'))
    except Exception: pass
    LOG(f'★이어달림 step {state["step"]} / epoch {state["epoch"]} / cursor {state.get("cursor", 0)} (옵티마이저·난수 상태는 아래에서 복원 시도)')
else:
    model = get_peft_model(model, LoraConfig(r=A.r, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROP, bias='none', target_modules=targets, task_type='CAUSAL_LM'))
model.print_trainable_parameters()

from torch.utils.checkpoint import checkpoint as _ckpt
_base = model.get_base_model(); _dec = _base.get_decoder() if hasattr(_base, 'get_decoder') else _base.model; _head = _base.get_output_embeddings()
def chunked_loss(ids, labels, attn):
    h = _dec(input_ids=ids, attention_mask=attn, use_cache=False).last_hidden_state
    n = ids.shape[1] - 1; h = h[:, :n]
    tot = h.new_zeros((), dtype=torch.float32); cnt = 0
    def _ce(hs, tg):
        lg = _head(hs).float()
        return torch.nn.functional.cross_entropy(lg.view(-1, lg.shape[-1]), tg.reshape(-1), ignore_index=-100, reduction='sum')
    for s in range(0, n, CE_CHUNK):
        hs, tg = h[:, s:s + CE_CHUNK], labels[:, s + 1:s + 1 + CE_CHUNK]
        c = int((tg != -100).sum())
        if c == 0: continue
        tot = tot + _ckpt(_ce, hs, tg, use_reentrant=False); cnt += c
    return tot / max(cnt, 1), cnt
def to_batch(e):
    ids = torch.tensor([e['input_ids']], device='cuda'); lab = torch.tensor([e['labels']], device='cuda')
    return ids, lab, torch.ones_like(ids)

import bitsandbytes as bnb_lib
params = [p for p in model.parameters() if p.requires_grad]
opt = bnb_lib.optim.PagedAdamW8bit(params, lr=A.lr, weight_decay=0.0)
RESUMED_FULL = False
if A.resume and os.path.exists(f'{CKPT}/optimizer.pt'):
    try:
        ck = torch.load(f'{CKPT}/optimizer.pt', map_location='cuda', weights_only=False)
        opt.load_state_dict(ck['opt']); torch.set_rng_state(ck['cpu_rng']); torch.cuda.set_rng_state(ck['cuda_rng']); random.setstate(ck['py_rng'])
        RESUMED_FULL = True; LOG('옵티마이저·난수 상태 복원')
    except Exception as e:
        LOG(f'★옵티마이저 상태 복원 실패 → 가중치만 이어달림(정확한 재개 아님): {repr(e)[:80]}')
steps_per_ep = math.ceil(len(train_ds) / GRAD_ACC); total_steps = 20 if A.smoke else steps_per_ep * A.epochs
warm = max(1, int(total_steps * 0.03))
def lr_at(s):
    if s < warm: return A.lr * (s + 1) / warm
    p = (s - warm) / max(1, total_steps - warm); return A.lr * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
os.makedirs(OUT_DIR, exist_ok=True)
def save_ckpt(tag='ckpt'):
    d = f'{OUT_DIR}/{tag}'; model.save_pretrained(d); json.dump(state, open(f'{d}/state.json', 'w'))
    try:
        torch.save({'opt': opt.state_dict(), 'cpu_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state(), 'py_rng': random.getstate()}, f'{d}/optimizer.pt')
    except Exception as e:
        LOG(f'   (옵티마이저 저장 실패: {repr(e)[:60]})')
    LOG(f'   저장 {d} (step {state["step"]})')
@torch.no_grad()
def evaluate():
    if not val_ds: return None
    model.eval(); tot = cnt = 0
    for e in val_ds:
        l, c = chunked_loss(*to_batch(e)); tot += l.item() * c; cnt += c
    model.train(); return tot / max(cnt, 1)

LOG(f'학습 시작 — {steps_per_ep} 스텝/epoch × {A.epochs} = {total_steps} (GRAD_ACC {GRAD_ACC}, r {A.r}, 가중 손실)')
model.train(); t0 = time.monotonic(); hist = []; step = state['step']; done = False
torch.cuda.reset_peak_memory_stats()
n_oom = 0
for ep in range(state['epoch'], A.epochs):
    order = list(range(len(train_ds))); random.Random(SEED + ep).shuffle(order)
    skip = state.get('cursor', 0) if ep == state['epoch'] else 0          # ★샘플 커서로 재개 (스텝 역산 아님)
    acc_loss = acc_n = 0; micro = 0                                          # micro = 이번 누적 묶음에서 성공한 microbatch 수
    for k in range(skip, len(order)):
        e = train_ds[order[k]]; ids, lab, att = to_batch(e)
        try:
            loss, c = chunked_loss(ids, lab, att)
            (loss * e['w']).backward()                                       # ★샘플 가중 (기전 균등화 × 출처); 누적 수로 나누는 건 스텝 때
        except torch.cuda.OutOfMemoryError:
            n_oom += 1; LOG(f'   ★OOM (len {ids.shape[1]}) — 이 샘플만 건너뜀 (누적 유지)'); torch.cuda.empty_cache(); gc.collect(); continue
        acc_loss += loss.item() * c; acc_n += c; micro += 1
        if micro == GRAD_ACC or k == len(order) - 1:
            for p in params:
                if p.grad is not None: p.grad.div_(micro)                     # ★실제 성공한 microbatch 수로 평균
            for g in opt.param_groups: g['lr'] = lr_at(step)
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); opt.zero_grad(set_to_none=True); micro = 0
            step += 1; state['step'] = step; state['epoch'] = ep; state['cursor'] = k + 1
            if step % LOG_EVERY == 0 or step == 1:
                LOG(f'ep {ep} step {step}/{total_steps} loss {acc_loss/max(acc_n,1):.4f} grad {gn.item():.2f} lr {lr_at(step):.2e} 피크 {torch.cuda.max_memory_allocated()/1e9:.1f}GB 경과 {(time.monotonic()-t0)/60:.0f}분')
                hist.append({'step': step, 'loss': acc_loss / max(acc_n, 1)})
            acc_loss = acc_n = 0
            if step % SAVE_EVERY == 0: save_ckpt()
            if step >= total_steps: done = True; break
    if val_ds:
        vl = evaluate(); LOG(f'★epoch {ep} val loss {vl:.4f}'); hist.append({'epoch': ep, 'val_loss': vl})
    state['epoch'] = ep + 1; state['cursor'] = 0; save_ckpt()
    if done: break

final = f'{OUT_DIR}/final'; model.save_pretrained(final); tok.save_pretrained(final)
meta = {'base_model': MODEL_ID, 'lib': L.LIB_VERSION, 'samples': len(train_ds), 'epochs': A.epochs, 'steps': step, 'lr': A.lr, 'lora_r': A.r,
        'max_len': MAX_LEN, 'targets_n': len(targets), 'peak_gb': round(torch.cuda.max_memory_allocated() / 1e9, 1),
        'minutes': round((time.monotonic() - t0) / 60), 'oom_skipped': n_oom, 'resumed_full': RESUMED_FULL, 'log': hist[-40:],
        'versions': {'transformers': __import__('transformers').__version__, 'peft': __import__('peft').__version__, 'torch': torch.__version__},
        'when': time.strftime('%Y-%m-%d %H:%M')}
json.dump(meta, open(f'{OUT_DIR}/train_meta.json', 'w'), ensure_ascii=False, indent=1)

# ---- PEFT 기준 출력 (적용 증명 재료) → 키 점검 → vLLM 용 리네임 사본 (기준 출력 포함) ----
model.eval(); refs = []
for r in (train_rows[:2] + (val_rows or [])[:1]):
    enc = tok(chat_prompt(r['prompt']), return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
    refs.append({'tid': r['tid'], 'split': 'train' if r in train_rows else 'val', 'prompt': r['prompt'], 'completion': r['completion'],
                 'peft_greedy': tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)})
json.dump(refs, open(f'{final}/ref_outputs.json', 'w'), ensure_ascii=False, indent=1)
rep = L.adapter_key_report(final); LOG('어댑터 키:', json.dumps(rep, ensure_ascii=False)[:400])
n_ren = L.rename_adapter(final, f'{OUT_DIR}/final_vllm'); LOG(f'리네임 사본 final_vllm (치환 {n_ren}키, ref_outputs 포함) — 셀8 적용증명이 원본에서 실패할 때만 쓴다')
print('=' * 76); print(f'★★★ 어댑터 {final} ({meta["minutes"]}분, 피크 {meta["peak_gb"]}GB)')
print('PEFT greedy 미리보기:', repr(refs[0]['peft_greedy'][:120]))
print('  다음: [런타임 재시작] → 셀1(STAGE=gen) → 셀2 → 셀8(사다리 평가)')


In [ ]:
# [셀8] ★사다리 평가 L1~L5 — base vs +어댑터, eval 홀드아웃 40 × K=8 서술 (STAGE='gen', 새 런타임)
#   ① 적용 증명: 고정 완성문 teacher-forcing 로그확률 (학습 샘플 3개) — lora 가 base 보다 높아야 한다. greedy 텍스트 차이는 참고만.
#   ② 두 arm 을 같은 배치에서 번갈아 (완료 태스크 집합 동일). 배치마다 저장.
#   ③ L1 형식 / L2 HEAD / L3 기전 recall·precision(사용자 119 기준) / L4 코드 복원(train 전쌍) / L5 oracle / ★L5_2slot(정답 비공개 투표 2슬롯) / 고유율
#   ④ 판정 2단: 연구 판정(L3·L4·고유율 → R1 진행 여부) 과 채택 판정(2슬롯 정답 태스크, 짝지은 승패) 을 분리
import os, sys, json, time, random, hashlib, glob
from collections import Counter
import numpy as np
sys.path.insert(0, '/content'); import importlib, desc_lora_lib as L; importlib.reload(L)
from desc_lora_lib import LOG

ADAPTER   = f'{DRIVE_ROOT}/lora_desc_q38/final'          # 적용증명 실패 시 → f'{DRIVE_ROOT}/lora_desc_q38/final_vllm'
MEMO_F    = f'{DESC_DIR}/eval119_사람서술_메모.txt'        # ★arc descript.txt 를 이 이름으로 복사해 둔다 (L3 기준)
N_HOLD, K, TEMP, RT_TRY = 40, 8, 0.8, 2
PROMPT_TOK_CAP, SEED, BUDGET_MIN = 6500, 42, 150
ENGINE_SOLVED = set('0934a4d8 67e490f4 6ffbe589 981571dc a251c730 a6f40cea e8686506 edb79dae fc7cae8d'.split())
DEV_TIDS = set()                # ★규칙·씨앗·프롬프트 개발에 참고한 eval tid — 수동 목록 + 아래 자동 검사(few-shot 출처)와 합집합
FAIL_AXIS = {'assemble', 'count', 'legend', 'hole'}
assert os.path.exists(f'{ADAPTER}/adapter_config.json'), f'어댑터 없음 {ADAPTER}'
rep = L.adapter_key_report(ADAPTER); LOG('어댑터 키:', json.dumps(rep, ensure_ascii=False)[:300])

EV = L.load_split('evaluation'); EV_IDS = [tid for tid, _ in EV]
HUMAN = L.eval119_by_tid(MEMO_F, EV_IDS) if os.path.exists(MEMO_F) else {}
LOG(f'사람 서술 매핑 {len(HUMAN)} (없으면 L3 생략)')
# 출처 자동 검사: 학습 데이터·few-shot 출처에 eval tid 가 섞였으면 제외 (있어선 안 되지만 검사한다)
_used = set()
for r in L.read_jsonl([f'{DESC_DIR}/desc_sft_train.jsonl', f'{DESC_DIR}/desc_sft_val.jsonl']):
    _used.add(r.get('tid')); _used |= {x for x in (r.get('fewshot_src') or []) if x}
leak = _used & set(EV_IDS)
if leak:
    LOG(f'★eval tid 가 학습/few-shot 출처에 있음 {sorted(leak)} → 홀드아웃에서 제외')
DEV_TIDS |= leak
ensure_model()
ENG = L.VEngine(LOCAL_DIR, max_model_len=32768, gpu_util=0.90, lora_dir=ADAPTER, max_lora_rank=max(64, rep['r'] or 64), seed=SEED)

# 홀드아웃
rng = random.Random(SEED); pool = []
for tid, t in EV:
    if tid in ENGINE_SOLVED or tid in DEV_TIDS or not L.usable(t):
        continue
    F = L.task_facts(t); p = L.prompt_desc(t, F)
    if ENG.n_tokens(p) <= PROMPT_TOK_CAP:
        pool.append((tid, t, F, p))
rng.shuffle(pool); HOLD = pool[:N_HOLD]; NTR = {tid: len(t['train']) for tid, t, _, _ in HOLD}
LOG(f'홀드아웃 {len(HOLD)} / 후보 {len(pool)}')
RUN_ID = time.strftime('%m%d_%H%M'); REPORT_F = f'{DESC_DIR}/ladder_{RUN_ID}.json'

# ① 적용 증명 — teacher forcing: 학습에 쓴 완성문의 로그확률이 lora 에서 올라가야 한다
refs = json.load(open(f'{ADAPTER}/ref_outputs.json')) if os.path.exists(f'{ADAPTER}/ref_outputs.json') else []
if not refs:
    refs = [{'tid': r['tid'], 'prompt': r['prompt'], 'completion': r['completion']} for r in L.read_jsonl([f'{DESC_DIR}/desc_sft_train.jsonl'])[:3]]
proof = []
for r in refs[:3]:
    lb, n = ENG.seq_logprob(r['prompt'], r['completion'], lora=False); ll, _ = ENG.seq_logprob(r['prompt'], r['completion'], lora=True)
    proof.append({'tid': r['tid'], 'n_tok': n, 'nll_base': round(-lb, 1), 'nll_lora': round(-ll, 1), 'lora_better': ll > lb + 1e-6})
gb, gl = ENG.greedy_logprob(HOLD[0][3], lora=False, max_tokens=48)[0], ENG.greedy_logprob(HOLD[0][3], lora=True, max_tokens=48)[0]
print('=' * 76); print('적용 증명 (teacher-forcing NLL, 낮을수록 좋음):'); [print(' ', json.dumps(x, ensure_ascii=False)) for x in proof]
print('  greedy 참고: base', repr(gb[:70]), '| lora', repr(gl[:70]))
if not proof or sum(x['lora_better'] for x in proof) * 2 < len(proof):
    raise SystemExit('★어댑터가 적용되지 않았거나 학습 분포를 못 배웠다 (학습 완성문 NLL 이 안 내려감). ADAPTER 를 final_vllm 로 바꿔 재실행. 그래도 같으면 키 매칭/vLLM 지원 문제 → 병합 경로')

# ② 두 arm 을 같은 배치에서 번갈아 — 완료 태스크 집합이 같다
def score_batch(batch, lora):
    texts = ENG.generate([p for _, _, _, p in batch], think=False, n=K, temp=TEMP, max_tokens=900, lora=lora, seed=SEED + hash(batch[0][0]) % 10000)
    rt_items, rt_meta, rows = [], [], []
    for (tid, t, F, p), row in zip(batch, texts):
        hm = L.human_mech_set(HUMAN[tid]) if tid in HUMAN else None
        recs = []
        for txt in row:
            d = L.parse_desc(txt)
            rec = {'L1': d is not None, 'L2': False, 'L3r': None, 'L3p': None, 'L4': False, 'L5': False, 'key': None, 'cands': []}
            if d is not None:
                rec['L2'] = L.head_ok(d, F); rec['key'] = L.desc_key(d); em = L.effective_mechs(d)
                if hm:
                    rec['L3r'] = len(em & hm) / len(hm); rec['L3p'] = (len(em & hm) / len(em)) if em else 0.0
                rt_items.append((t, d['block'])); rt_meta.append(rec)
            recs.append(rec)
        rows.append({'tid': tid, 'n_train': len(t['train']), 'human_mechs': sorted(hm) if hm else None, 'recs': recs})
    for rec, r in zip(rt_meta, L.roundtrip_batch(ENG, rt_items, n_try=RT_TRY)):
        rec['L5'] = r['ok']; rec['hist'] = r['hist']; rec['cands'] = [{'tp': c['tp'], 'te': c['te'], 'preds': c['preds']} for c in r['cands']]
    for row in rows:
        allc = [c for rec in row['recs'] for c in rec['cands']]
        for rec in row['recs']:
            rec['L4'] = any(c['tp'] == row['n_train'] for c in rec['cands'])
        slots = L.pick_two_slots(allc, row['n_train'])                      # ★정답 비공개 선택
        sol = dict(HOLD_T[row['tid']])['_sol']
        row['slot2_correct'] = any(len(s) == len(sol) and all(np.array_equal(np.asarray(a), np.asarray(b)) for a, b in zip(s, sol)) for s in slots)
        row['n_slots'] = len(slots); row['oracle'] = any(rec['L5'] for rec in row['recs'])
        for rec in row['recs']:
            rec.pop('cands', None)                                          # 리포트 크기
    return rows

HOLD_T = {tid: t for tid, t, _, _ in HOLD}
RES = {'base': [], 'lora': []}; DEAD = time.monotonic() + BUDGET_MIN * 60
for b0 in range(0, len(HOLD), 8):
    if time.monotonic() > DEAD:
        LOG('예산 마감 — 두 arm 모두 같은 태스크까지'); break
    batch = HOLD[b0:b0 + 8]
    for arm in ('base', 'lora'):
        t0 = time.monotonic(); RES[arm] += score_batch(batch, arm == 'lora')
        LOG(f'  {arm:4s} {b0+len(batch)}/{len(HOLD)} ({round(time.monotonic()-t0)}s)')
    json.dump({'adapter': ADAPTER, 'lib': L.LIB_VERSION, 'K': K, 'rt_try': RT_TRY, 'proof': proof, 'rows': RES, 'done_tids': [b[0] for b in HOLD[:b0+len(batch)]]},
              open(REPORT_F, 'w'), ensure_ascii=False, indent=1)

def summarize(rows, axis=None):
    n = l1 = l2 = l4 = l5 = 0; l3r, l3p, uniq = [], [], []; oracle = slot2 = tasks = 0
    for r in rows:
        if axis and not (r['human_mechs'] and set(r['human_mechs']) & axis):
            continue
        tasks += 1; keys = set()
        for rec in r['recs']:
            n += 1; l1 += rec['L1']; l2 += rec['L2']; l4 += rec['L4']; l5 += rec['L5']
            if rec['L3r'] is not None: l3r.append(rec['L3r']); l3p.append(rec['L3p'])
            if rec['key']: keys.add(rec['key'])
        uniq.append(len(keys) / max(1, len(r['recs']))); oracle += r['oracle']; slot2 += r['slot2_correct']
    if n == 0: return None
    return {'tasks': tasks, 'desc': n, 'L1_format': round(l1 / n, 3), 'L2_head': round(l2 / n, 3),
            'L3_recall': round(float(np.mean(l3r)), 3) if l3r else None, 'L3_precision': round(float(np.mean(l3p)), 3) if l3p else None,
            'L4_train_pass': round(l4 / n, 3), 'L5_oracle_tasks': oracle, 'L5_2slot_tasks': slot2, 'uniq_rate': round(float(np.mean(uniq)), 3)}

print('=' * 76)
S = {arm: summarize(RES[arm]) for arm in RES}; SA = {arm: summarize(RES[arm], FAIL_AXIS) for arm in RES}
for arm in ('base', 'lora'):
    print(f'[{arm:4s}] 전체  ', json.dumps(S[arm])); print(f'[{arm:4s}] 실패축', json.dumps(SA[arm]))
b, l = S['base'], S['lora']
paired = {r['tid']: r['slot2_correct'] for r in RES['base']}
won = [r['tid'] for r in RES['lora'] if r['slot2_correct'] and not paired.get(r['tid'])]
lost = [r['tid'] for r in RES['lora'] if not r['slot2_correct'] and paired.get(r['tid'])]
# 연구 판정 (R1 진행 여부): 서술 수준 지표
has_l3 = l['L3_recall'] is not None and b['L3_recall'] is not None
gain = (l['L3_recall'] - b['L3_recall'] > 0.05 and l['L3_precision'] >= b['L3_precision'] - 0.05) if has_l3 else (l['L4_train_pass'] - b['L4_train_pass'] > 0.05)
if gain and l['L4_train_pass'] >= b['L4_train_pass'] and l['uniq_rate'] >= 0.7 * b['uniq_rate'] and len(lost) <= len(won) + 1:
    research = 'ACCEPT'
elif l['L4_train_pass'] < 0.85 * b['L4_train_pass'] or l['uniq_rate'] < 0.6 * b['uniq_rate'] or len(lost) > len(won) + 2:
    research = 'REJECT'
else:
    research = 'TIE'
# 채택 판정 (제출 후보): 2슬롯 정답 태스크 짝지은 승패 — 표본이 작으면 결론 유보
adopt = 'CANDIDATE' if (len(won) - len(lost) >= 3) else ('NO' if len(lost) - len(won) >= 3 else 'UNDECIDED')
verdict = {'research': research, 'adopt': adopt, 'won': won, 'lost': lost, 'n_tasks': len(RES['base'])}
rep_all = json.load(open(REPORT_F)); rep_all.update({'summary': S, 'fail_axis': SA, 'verdict': verdict, 'final': True}); json.dump(rep_all, open(REPORT_F, 'w'), ensure_ascii=False, indent=1)
print('-' * 76)
print(f"★연구 판정: {research}  (L3 recall +0.05 & precision 유지 & L4 유지 & 고유율 ≥70% & 2슬롯 잃음 ≤ 얻음+1)")
print(f"★채택 판정: {adopt}  (2슬롯 정답 태스크 base {b['L5_2slot_tasks']} → lora {l['L5_2slot_tasks']}, 얻음 {won} / 잃음 {lost}; ±3 미만은 UNDECIDED)")
print(f"  oracle(16후보 중 정답 존재) base {b['L5_oracle_tasks']} → lora {l['L5_oracle_tasks']} 는 상한 지표일 뿐 제출 성적이 아님. 리포트: {REPORT_F}")
if research == 'ACCEPT':
    print('  다음: 셀3 LORA_DIR=ADAPTER 로 R1 재채굴 → 셀6 → 셀7 → v2. 채택은 동일 시간 예산의 리허설(처리량 포함)에서 다시 판단.')
